In [1]:
import datetime
import calendar

# Obtener la fecha actual
hoy = datetime.date.today()

# Calcular el primer día del mes actual y restar un día para obtener el mes anterior
primer_dia_mes_actual = hoy.replace(day=1)
ultimo_dia_mes_anterior = primer_dia_mes_actual - datetime.timedelta(days=1)

# Extraer mes y año del mes anterior
mes_anterior = ultimo_dia_mes_anterior.month
anio_anterior = ultimo_dia_mes_anterior.year

# Asignar variables
MES_ACTUAL_EN_LIQUIDACION = f"'{anio_anterior}{mes_anterior:02d}'"

# Imprimir resultados
print("Mes actual =", MES_ACTUAL_EN_LIQUIDACION)

Mes actual = '202507'


In [2]:
#Importacion de librerias
#import requests
from sqlalchemy import create_engine, text, inspect
from sqlalchemy.orm import sessionmaker
import pandas as pd
from unidecode import unidecode
import numpy as np
#from tqdm import tqdm
import json
from pandas import json_normalize
import openpyxl
import concurrent.futures

Extraccion del proceso de etiquetacion despues del penultimo paso

In [3]:
# Configura los detalles de la conexión
host='172.16.1.33'
port='1433'
database= 'CUN_REPOSITORIO'

try:
    # Crea la cadena de conexión
   url = 'mssql+pyodbc://@{host}:{port}/{db}?driver=SQL+Server'.format(host=host, port=port, db=database)

   
    # Crear conexion con base de datos
   engine = create_engine(url) 
   print("Conexion a la base de datos realizada") 
    
except Exception as e:
    print("Error:", e)

Conexion a la base de datos realizada


In [4]:
#CONSULTA PARA CREAR LA TABLA TEMPORAL EN SQL 
with engine.connect() as connection:
    try:

        sql_query = text('''
         IF object_id('CUN_REPOSITORIO.COMISIONES_CONTACT.TEMP_ZOHO_FUNELL') IS NOT NULL
    DROP TABLE CUN_REPOSITORIO.COMISIONES_CONTACT.TEMP_ZOHO_FUNELL

        select *,
            case
            when Propietario_de_Fase is null then Propietario
            else Propietario_de_Fase end as
            Propietario_de_Fase_Name
            into CUN_REPOSITORIO.COMISIONES_CONTACT.TEMP_ZOHO_FUNELL
        from(
                        select
                            Periodo
                            ,Número_de_Documento
                            ,Correo_electronico_asesor,
                            case when Propietario_de_Oportunidad_Name like '%@%' then substring(Propietario_de_Oportunidad_Name,CHARINDEX(' ', Propietario_de_Oportunidad_Name)+1,50)
                            else Correo_electronico_asesor end Propietario_de_Fase,
                            case when B.Nombre_completo like '%@%' then substring(B.Nombre_completo,CHARINDEX(' ', B.Nombre_completo)+1,50)
                            else B.Correo_electrónico end Propietario,
                            row_number() over (partition by Número_de_Documento,Periodo  order by FechaCreación2 desc ) cont
                        from CUN_REPOSITORIO.dbo.zoho_Funell A
                        left join zoho.CRM.Usuarios b on a.Propietario_de_Oportunidad = b.ID
                        )t1
        ''')
        connection.execute(sql_query)
        connection.commit()  # Asegurar que los cambios se guarden en la base de datos
        print("Consulta realizada con exito")
    except Exception as e:
        print(f"Error al eliminar registros: {e}")
    finally:
        connection.close()

Consulta realizada con exito


In [5]:
##CONSULTA PARA CREAR LA TABLA TEMPORAL EN SQL 
with engine.connect() as connection:
    try:

        sql_query = text('''
         IF object_id('CUN_REPOSITORIO.COMISIONES_CONTACT.TEMP_ZOHO_FUNELL') IS NOT NULL
        DROP TABLE CUN_REPOSITORIO.COMISIONES_CONTACT.TEMP_ZOHO_FUNELL

        select *,
            case
            when Propietario_de_Fase is null then Propietario
            else Propietario_de_Fase end as
            Propietario_de_Fase_Name
            into CUN_REPOSITORIO.COMISIONES_CONTACT.TEMP_ZOHO_FUNELL
        from(
                        select
                            Periodo
                            ,Número_de_Documento
                            ,Correo_electronico_asesor,
                            case when Propietario_de_Oportunidad_Name like '%@%' then substring(Propietario_de_Oportunidad_Name,CHARINDEX(' ', Propietario_de_Oportunidad_Name)+1,50)
                            else Correo_electronico_asesor end Propietario_de_Fase,
                            case when B.Nombre_completo like '%@%' then substring(B.Nombre_completo,CHARINDEX(' ', B.Nombre_completo)+1,50)
                            else B.Correo_electrónico end Propietario,
                            row_number() over (partition by Número_de_Documento,Periodo  order by FechaCreación2 desc ) cont
                        from CUN_REPOSITORIO.dbo.zoho_Funell A
                        left join zoho.CRM.Usuarios b on a.Propietario_de_Oportunidad = b.ID
                        )t1
        ''')
        connection.execute(sql_query)
        connection.commit()  # Asegurar que los cambios se guarden en la base de datos
        print("Consulta realizada con exito")
    except Exception as e:
        print(f"Error al eliminar registros: {e}")
    finally:
        connection.close()

Consulta realizada con exito


In [6]:
##CONSULTA PARA CREAR LA TABLA TEMPORAL EN SQL 
with engine.connect() as connection:
    try:

        sql_query = text('''
         IF object_id('[tempdb]..[##Ventas]') IS NOT NULL
            drop table ##Ventas

                    SELECT
                          *
                    into ##Ventas
                    FROM OPENQUERY([172.16.1.175],
                    'select distinct * from iceberg.VEN_VENTAS where ANIO >= ''2025'' ')
        ''')
        connection.execute(sql_query)
        connection.commit()  # Asegurar que los cambios se guarden en la base de datos
        print("Consulta realizada con exito")
    except Exception as e:
        print(f"Error al eliminar registros: {e}")
    finally:
        connection.close()

Consulta realizada con exito


In [7]:
##CONSULTA PARA CREAR LA TABLA TEMPORAL EN SQL 
with engine.connect() as connection:
    try:

        sql_query = text(f'''
         IF object_id('CUN_REPOSITORIO.COMISIONES_CONTACT.TEMP_BASE_VENTAS') IS NOT NULL
        DROP TABLE CUN_REPOSITORIO.COMISIONES_CONTACT.TEMP_BASE_VENTAS
                         
            Select t1.*,t2.PERIODO as Periodo_Pago,t3.EST_MATRICULADO,t3.NCR_ANULACION,t3.TIPO,t3.ORDEN,
                t3.VAL_LIQUIDADO,t3.NUEVO,t3.ESTADO_PAGO,t3.Doc_Alum as Documento_Estudiante_zoho
                Into CUN_REPOSITORIO.COMISIONES_CONTACT.TEMP_BASE_VENTAS
                from Comisiones.Ventas_Contact T1
            left join CUN_REPOSITORIO.cun.Calendario_Anual T2 ON CONVERT(date, t1.Fecha_de_pago, 103) = CONVERT(date, t2.FECHA_MES, 103)
            Left join (Select * from (
            select Doc_Alum,ESTADO_PAGO,NCR_ANULACION,NUEVO,EST_MATRICULADO,PERIODO,ORDEN,VAL_LIQUIDADO,TIPO
                    , row_number() over (partition by Doc_Alum,PERIODO  order by ESTADO_PAGO desc,NCR_ANULACION desc, EST_MATRICULADO,ORDEN asc ) cont
                                from ##Ventas 	where DOCUMENTO in ( 'FAMA','FECO')  and PERIODO not like '%I%'
                                                                                )t1
                                where cont = 1) t3 on t1.Doc_Estudiante = t3.Doc_Alum and t1.PERIODO = t3.PERIODO
            where t2.PERIODO = {MES_ACTUAL_EN_LIQUIDACION}
        ''')
        connection.execute(sql_query)
        connection.commit()  # Asegurar que los cambios se guarden en la base de datos
        print("Consulta realizada con exito")
    except Exception as e:
        print(f"Error al eliminar registros: {e}")
    finally:
        connection.close()

Consulta realizada con exito


In [8]:
##CONSULTA PARA CREAR LA TABLA TEMPORAL EN SQL 
with engine.connect() as connection:
    try:

        sql_query = text(f'''
        
                Delete from  Comisiones_Contact.Etiquetas where Periodo_Pago = {MES_ACTUAL_EN_LIQUIDACION}

                INSERT INTO Comisiones_Contact.Etiquetas
                select *,
                            case
                            when Motivo =  'OK' THEN 'OK'
                            ELSE Motivo END AS EstadoFinal
                            from (select distinct t1.*,
                                            case
                                                WHEN T3.Doc_Estudiante IS NOT NULL THEN 'YA SE PAGO'
                                                when t1.Documento_Estudiante_zoho is null
                                                then 'No coincide con Iceberg o Sinu'
                                                when t1.ESTADO_PAGO <> 'PAGO' then 'No Registra Pago'
                                                when t1.ESTADO_PAGO IS NULL then 'No Registra Pago'
                                                when t1.EST_MATRICULADO IS NULL then 'No Registra Matricula'
                                                --  when t1.NCR_ANULACION < 0 then 'Tiene Anulación'
                                                when t1.NUEVO = 'ANTIGUO' and TIPO <> 'Educacion continuada' then 'Estudiante ANTIGUO'
                                                when t1.EST_MATRICULADO <> '1-Activo'
                                                then 'Estado Diferente al Activo ' + t1.EST_MATRICULADO
                                                when t1.VAL_LIQUIDADO < 200000 then 'Valor liquidado Menor 20%'
                                                --when t4.CorreoPersonalSugerido is null then 'Sin Correo Electronico'
                                                --when t4.CelularSugerido is null then 'Sin Telefono personal'
                                                else 'ok' end Motivo,
                                            t2.Propietario_de_Fase_Name--,t2.Correo_electronico_asesor
                                        from CUN_REPOSITORIO.COMISIONES_CONTACT.TEMP_BASE_VENTAS t1
                                        left join CUN_REPOSITORIO.COMISIONES_CONTACT.TEMP_ZOHO_FUNELL t2
                                                on t1.Doc_Estudiante = t2.Número_de_Documento and t1.PERIODO = t2.Periodo and
                                                    t1.Correo_Asesor = t2.Propietario_de_Fase_Name
                                        LEFT JOIN (select distinct Doc_Estudiante,PERIODO from Comisiones_Contact.Etiquetas where EstadoFinal = 'OK') T3
                                                ON T1.Doc_Estudiante = T3.Doc_Estudiante AND T1.PERIODO = T3.PERIODO
                                        left join (select * from [Calidad].[Datos_contacto_CUN2]) t4
                                                on t1.Documento_Estudiante_zoho = t4.NUM_IDENTIFICACION

                            )t1

        ''')
        connection.execute(sql_query)
        connection.commit()  # Asegurar que los cambios se guarden en la base de datos
        print("Registros eliminados correctamente.")
    except Exception as e:
        print(f"Error al eliminar registros: {e}")
    finally:
        connection.close()

Registros eliminados correctamente.


In [9]:
host2_1 = '172.16.1.33'
port2_1 = '1433'
database2_1 = 'CUN_REPOSITORIO'

try:
    url2_1 = 'mssql+pyodbc://@{host}:{port}/{db}?trusted_connection=yes&driver=SQL+Server'.format(host=host2_1, port=port2_1, db=database2_1)
    engine2_1 = create_engine(url2_1) 
    print("Conexion a la base de datos realizada")
    #archivo_orig = pd.read_sql("select * from CUN_REPOSITORIO.Comisiones.Com_Telec_Oct_2024_2_yCT", engine2_1)
    Etiquetas_penultimo_paso = pd.read_sql(f'''
        select * from CUN_REPOSITORIO.COMISIONES_CONTACT.ETIQUETAS
            WHERE Periodo_Pago ={MES_ACTUAL_EN_LIQUIDACION} and EstadoFinal ='OK'
                               ''', engine2_1)
    Consulta_z2 = Etiquetas_penultimo_paso.copy()
        
except Exception as e:
    print("Error:", e)
finally:
    # Cerrar la conexión
    if 'engine2_1' in locals():
        engine2_1.dispose()
        print("Conexión cerrada")
#-------------------------------------------------------------------------------------------
Etiquetas_penultimo_paso['Doc_Estudiante'] = Etiquetas_penultimo_paso['Doc_Estudiante'].astype(int).astype(str)
Etiquetas_penultimo_paso['PERIODO'] = Etiquetas_penultimo_paso['PERIODO'].astype(str)
Etiquetas_penultimo_paso['Llave_per'] = Etiquetas_penultimo_paso['Doc_Estudiante'] + Etiquetas_penultimo_paso['PERIODO']
#--------------------------------------------------------------------------------------------
Etiquetas_penultimo_paso.shape
#--------------------------------------------------------------------------------------------
#--------------------------------------------------------------------------------------------

Conexion a la base de datos realizada
Conexión cerrada


(3477, 21)

In [10]:
##CONSULTA PARA CREAR LA TABLA TEMPORAL EN SQL 
with engine.connect() as connection:
    try:

        sql_query = text(f'''
         DELETE FROM CUN_REPOSITORIO.COMISIONES_CONTACT.ETIQUETAS WHERE Periodo_Pago = {MES_ACTUAL_EN_LIQUIDACION} and EstadoFinal ='OK'
        ''')
        connection.execute(sql_query)
        connection.commit()  # Asegurar que los cambios se guarden en la base de datos
        print("Registros eliminados correctamente.")
    except Exception as e:
        print(f"Error al eliminar registros: {e}")
    finally:
        connection.close()

Registros eliminados correctamente.


# CONSULTA TOTAL y tratamiento general

In [11]:
para_orden = Etiquetas_penultimo_paso['Doc_Estudiante'].unique()
valores_unicos_df1_orden = "', '".join(map(str, para_orden))
valores_unicos_df1_orden = f"'{valores_unicos_df1_orden}'"
#----------------------------------------------------------------------------------
host2_1 = '172.16.1.33'
port2_1 = '1433'
database2_1 = 'CUN_REPOSITORIO'

try:
    url2_1 = 'mssql+pyodbc://@{host}:{port}/{db}?trusted_connection=yes&driver=SQL+Server'.format(host=host2_1, port=port2_1, db=database2_1)
    engine2_1 = create_engine(url2_1) 
    print("Conexion a la base de datos realizada")
    df_ordenes= pd.read_sql(f'''
SELECT DISTINCT Documento_Estudiante_zoho, 
    PERIODO, 
    COD_UNI,
    FuerzaComercialFinal,
    NUEVO,
    CLASE_ACTUAL
FROM CUN_REPOSITORIO.DBO.RECIBOS_CONTACT2_SIN_ZOHO with (nolock)
     WHERE Documento_Estudiante_zoho IN ({valores_unicos_df1_orden})
       AND ESTADO_PAGO = 'PAGO'
       AND EST_MATRICULADO = '1-Activo'
                            ''', engine2_1)
    df_ordenes2 = df_ordenes.copy()
        
except Exception as e:
    print("Error:", e)
finally:
    # Cerrar la conexión
    if 'engine2_1' in locals():
        engine2_1.dispose()
        print("Conexión cerrada")
df_ordenes['llave_orden_sql']=df_ordenes['Documento_Estudiante_zoho'].astype(str) + df_ordenes['PERIODO'].astype(str)
df_ordenes['llave_total_cod_uni']=df_ordenes['Documento_Estudiante_zoho'].astype(str) + df_ordenes['COD_UNI'].astype(str)

Conexion a la base de datos realizada
Conexión cerrada


In [12]:
#ARMO UNA MINI CONSULTA PERO SOLO PARA VALIDAR COD_UNI EN EL PERIODO ACTUAL
df_recibos_per_Actual=df_ordenes[df_ordenes['llave_orden_sql'].isin(Etiquetas_penultimo_paso['Llave_per'])]
df_recibos_per_Actual=df_recibos_per_Actual.drop_duplicates(subset='llave_orden_sql', keep='first')


Me traigo el codigo de unidad a mi liquidacion actual

In [13]:
Etiquetas_penultimo_paso = Etiquetas_penultimo_paso.merge(df_recibos_per_Actual[['llave_orden_sql', 'COD_UNI']], 
                                                left_on='Llave_per', 
                                                right_on='llave_orden_sql', 
                                                how='left')
Etiquetas_penultimo_paso.drop(columns=['llave_orden_sql'], inplace=True)
Etiquetas_penultimo_paso.shape

(3477, 22)

# VALIDACION VS HISTORICO TELECAMPUS

In [14]:
para_orden = Etiquetas_penultimo_paso['Doc_Estudiante'].unique()
valores_unicos_df1_orden = "', '".join(map(str, para_orden))
valores_unicos_df1_orden = f"'{valores_unicos_df1_orden}'"
#----------------------------------------------------------------------------------
host2_1 = '172.16.1.33'
port2_1 = '1433'
database2_1 = 'CUN_REPOSITORIO'

try:
    url2_1 = 'mssql+pyodbc://@{host}:{port}/{db}?trusted_connection=yes&driver=SQL+Server'.format(host=host2_1, port=port2_1, db=database2_1)
    engine2_1 = create_engine(url2_1) 
    print("Conexion a la base de datos realizada")
    df_histo_telec= pd.read_sql(f'''
SELECT DISTINCT DOC_ALUM, PERIODO, COD_UNI FROM CUN_REPOSITORIO.FINANCIERA.HISTORICO_COM_TELEC_2025
     WHERE DOC_ALUM IN ({valores_unicos_df1_orden})
                            ''', engine2_1)
        
except Exception as e:
    print("Error:", e)
finally:
    # Cerrar la conexión
    if 'engine2_1' in locals():
        engine2_1.dispose()
        print("Conexión cerrada")
df_histo_telec['llave_hist_tel_cod_uni']=df_histo_telec['DOC_ALUM'].astype(str) + df_histo_telec['COD_UNI'].astype(str)
df_histo_telec['llave_hist_CC_PER']=df_histo_telec['DOC_ALUM'].astype(str) + df_histo_telec['PERIODO'].astype(str)

df_histo_telec
#--------------------------------
#--------------------------------

Conexion a la base de datos realizada
Conexión cerrada


,DOC_ALUM,PERIODO,COD_UNI,llave_hist_tel_cod_uni,llave_hist_CC_PER
0,1000519674,25V04,VTE01,1000519674VTE01,100051967425V04
1,1001289262,25ET3,ESP11,1001289262ESP11,100128926225ET3
2,1003518776,25V04,VP15G,1003518776VP15G,100351877625V04
3,1005827644,25V04,VTC09,1005827644VTC09,100582764425V04
4,1007395169,25ES2,ESP20,1007395169ESP20,100739516925ES2
5,1007562919,25V04,VTM62,1007562919VTM62,100756291925V04
6,1010224513,25V04,VTC06,1010224513VTC06,101022451325V04
7,1012412522,25V04,VP15G,1012412522VP15G,101241252225V04
8,1013683141,25V04,VTE01,1013683141VTE01,101368314125V04
9,1015457096,25V04,VTC09,1015457096VTC09,101545709625V04


Validacion VS TELECAMPUS directo POR PERIODO

In [15]:
Etiquetas_penultimo_paso['Esta_en_hist_telec_x_per']=Etiquetas_penultimo_paso['Llave_per'].isin(df_histo_telec['llave_hist_CC_PER']).astype(int)
Etiquetas_penultimo_paso.loc[Etiquetas_penultimo_paso['Esta_en_hist_telec_x_per']==1, 'EstadoFinal'] = 'ENTRO CON OTRA FUERZA_T'
#----------------------------------

Validacion VS TELECAMPUS POR CODIGO DE UNIDAD

In [16]:
#ARMO LLAVE POR COD UNI PARA TODOS
Etiquetas_penultimo_paso['Llave_liq_cod_uni']=Etiquetas_penultimo_paso['Doc_Estudiante'] + Etiquetas_penultimo_paso['COD_UNI']

"Divido mi liquidacion entre los que si obtuvieron su cod_uni y los que no"
Etiquetas_penultimo_paso_sin_cod_uni=Etiquetas_penultimo_paso[Etiquetas_penultimo_paso['COD_UNI'].isna()]
Etiquetas_penultimo_Con_cod_uni=Etiquetas_penultimo_paso[Etiquetas_penultimo_paso['COD_UNI'].notna()]

In [17]:
condicion_histor_telec_novedades=(
    (df_histo_telec['COD_UNI'].isna())|
    (df_histo_telec['COD_UNI']=='0')|
    (df_histo_telec['COD_UNI']==0)|
    (df_histo_telec['COD_UNI']=='Null')|
    (df_histo_telec['COD_UNI']=='null')|
    (df_histo_telec['COD_UNI']=='NULL')|
    (df_histo_telec['COD_UNI']=='')|
    (df_histo_telec['COD_UNI']==' ')|
    (df_histo_telec['COD_UNI']=='na')    
)

#divido mi historico telecampus en dos, que tiene COD_UNI Valido y los que no
df_histo_telec_SOLO_NOvedades=df_histo_telec[condicion_histor_telec_novedades]
df_histo_telec_Depurado=df_histo_telec[~condicion_histor_telec_novedades]

#ETIQUETEO MI LIQUIDACION CON COD_UNI
# VALIDO CONTRA EL HISTORICO telec de novedades por cc y contra el historico tele depurado por llave
Etiquetas_penultimo_Con_cod_uni['Esta_En_histor_telec_depur']=Etiquetas_penultimo_Con_cod_uni['Llave_liq_cod_uni'].isin(df_histo_telec_Depurado['llave_hist_tel_cod_uni']).astype(int)
Etiquetas_penultimo_Con_cod_uni['Esta_En_histor_telec_NOved']=Etiquetas_penultimo_Con_cod_uni['Doc_Estudiante'].isin(df_histo_telec_SOLO_NOvedades['DOC_ALUM']).astype(int)

condicion_cod_uni_liq=((Etiquetas_penultimo_Con_cod_uni['Esta_En_histor_telec_depur']==1)|(Etiquetas_penultimo_Con_cod_uni['Esta_En_histor_telec_NOved']==1))
Etiquetas_penultimo_Con_cod_uni.loc[condicion_cod_uni_liq, 'EstadoFinal'] = 'ENTRO CON OTRA FUERZA_T'

#ETIQUETO LOS QUE NO TIENEN COD UNI
Etiquetas_penultimo_paso_sin_cod_uni['Esta_En_histor_telec_TOTAL']=Etiquetas_penultimo_paso_sin_cod_uni['Doc_Estudiante'].isin(df_histo_telec['DOC_ALUM']).astype(int)
Etiquetas_penultimo_paso_sin_cod_uni.loc[Etiquetas_penultimo_paso_sin_cod_uni['Esta_En_histor_telec_TOTAL']==1, 'EstadoFinal'] = 'ENTRO CON OTRA FUERZA_T'

#VUELVO A UNIR MI LIQUIDACION
Etiquetas_penultimo_paso=[]
Etiquetas_penultimo_paso = pd.concat([Etiquetas_penultimo_Con_cod_uni, Etiquetas_penultimo_paso_sin_cod_uni], ignore_index=True)
#--------------------------------------------
#--------------------------------------------

# COMIENZA LA VALIDACION DE NUEVOS - POR PERIODO ACTUAL

In [18]:
# CREO UNA CONSULTA CON SOLO DONDE NUEVO = NUEVO Y CLASE ACTUAL= NUEVO
Consulta_recibos_per_Actual_solo_nuevos=df_recibos_per_Actual[(df_recibos_per_Actual['NUEVO']=='NUEVO')&(df_recibos_per_Actual['CLASE_ACTUAL']=='Nuevo')]
Consulta_recibos_per_Actual_solo_nuevos.shape

#AHORA SIMPLEMENTE COMPARO SI MIS LIQUIDACION ACTUAL EN SU RESPECTIVO PERIODO SI ES NUEVO
Etiquetas_penultimo_paso['Esta_en_consulta_Nuevos'] =Etiquetas_penultimo_paso['Llave_per'].isin(Consulta_recibos_per_Actual_solo_nuevos['llave_orden_sql']).astype(int)
Etiquetas_penultimo_paso['Esta_en_consulta_per_Actual'] =Etiquetas_penultimo_paso['Llave_per'].isin(df_recibos_per_Actual['llave_orden_sql']).astype(int)

condicion_real_antiguo=((Etiquetas_penultimo_paso['Esta_en_consulta_Nuevos']==0) & (Etiquetas_penultimo_paso['Esta_en_consulta_per_Actual']==1))
Etiquetas_penultimo_paso.loc[condicion_real_antiguo, 'EstadoFinal'] = 'Estudiante ANTIGUO_PER_ACT'

# VALIDACION FUERZA DE VENTA POR PERIODO ACTUAL

In [19]:
# CREO UNA CONSULTA DONDE HAYAN FUERZAS DE VENTA VALIDAS Y LUEGO DONDE SOLO SEA EL CONTACT
Consulta_recibos_per_Actual_varias_fuerzas=df_recibos_per_Actual.copy()
Consulta_recibos_per_Actual_varias_fuerzas=Consulta_recibos_per_Actual_varias_fuerzas[Consulta_recibos_per_Actual_varias_fuerzas['FuerzaComercialFinal']!= 'N/A']
Consulta_recibos_per_Actual_varias_fuerzas=Consulta_recibos_per_Actual_varias_fuerzas[Consulta_recibos_per_Actual_varias_fuerzas['FuerzaComercialFinal']!= 'NA']
Consulta_recibos_per_Actual_varias_fuerzas=Consulta_recibos_per_Actual_varias_fuerzas[Consulta_recibos_per_Actual_varias_fuerzas['FuerzaComercialFinal']!= 'PENDIENTE']
Consulta_recibos_per_Actual_varias_fuerzas=Consulta_recibos_per_Actual_varias_fuerzas[Consulta_recibos_per_Actual_varias_fuerzas['FuerzaComercialFinal'].notna()]

#Creo df solo con CONTACT
Consulta_recibos_per_Actual_solo_CONTACT=Consulta_recibos_per_Actual_varias_fuerzas[(Consulta_recibos_per_Actual_varias_fuerzas['FuerzaComercialFinal']=='CONTACT')|(Consulta_recibos_per_Actual_varias_fuerzas['FuerzaComercialFinal']=='CONTACT ')]

#AHORA SIMPLEMENTE COMPARO SI MIS LIQUIDACION ACTUAL EN SU RESPECTIVO PERIODO SI ES NUEVO
Etiquetas_penultimo_paso['Esta_en_consulta_CONTACT'] =Etiquetas_penultimo_paso['Llave_per'].isin(Consulta_recibos_per_Actual_solo_CONTACT['llave_orden_sql']).astype(int)
Etiquetas_penultimo_paso['Esta_en_consulta_per_Actual_total_fuerzas'] =Etiquetas_penultimo_paso['Llave_per'].isin(Consulta_recibos_per_Actual_varias_fuerzas['llave_orden_sql']).astype(int)

condicion_OTRA_fuerza=((Etiquetas_penultimo_paso['Esta_en_consulta_CONTACT']==0) & (Etiquetas_penultimo_paso['Esta_en_consulta_per_Actual_total_fuerzas']==1))
Etiquetas_penultimo_paso.loc[condicion_OTRA_fuerza, 'EstadoFinal'] = 'OTRA FUERZA DE VENTA_PER_ACT'

# VALIDACION FUERZA DE VENTA - POR PRIMER PERIODO (COD UNI)

In [20]:
"Nuevamente Divido mi liquidacion entre los que si obtuvieron su cod_uni y los que no"
Etiquetas_penultimo_paso_sin_cod_uni=[]
Etiquetas_penultimo_Con_cod_uni=[]
Etiquetas_penultimo_paso_sin_cod_uni=Etiquetas_penultimo_paso[Etiquetas_penultimo_paso['Llave_liq_cod_uni'].isna()]
Etiquetas_penultimo_Con_cod_uni=Etiquetas_penultimo_paso[Etiquetas_penultimo_paso['Llave_liq_cod_uni'].notna()]

In [21]:
# AQUI SOLO ARMO UNA CONSULTA MAS GLOBAL no por periodo SINO con los CODIGOS DE unidad (contemplando todos los periodos) con los programas
# de la liquidacion
Consulta_total_cod_uni=df_ordenes[df_ordenes['llave_total_cod_uni'].isin(Etiquetas_penultimo_Con_cod_uni['Llave_liq_cod_uni'])]

# Dejo en la consultA
Consulta_total_cod_uni = Consulta_total_cod_uni.sort_values(by=['llave_total_cod_uni', 'PERIODO'], ascending=[True, True])
#------------------EXPORTANDO EVIDENCIA-------------------------------------
from datetime import datetime
fecha_hoy_rc = datetime.now().strftime('%Y-%m-%d')
Consulta_total_cod_uni.to_excel(f'Evidencia_Recibos_cont_CODUNI{fecha_hoy_rc}.xlsx')
#-------------------------------------------------
Consulta_total_cod_uni = Consulta_total_cod_uni.drop_duplicates(subset='llave_total_cod_uni', keep='first')
# Justo en este momento tengo la consulta lista con los primeros periodos
Consulta_solo_para_validar_nuevo_por_uni=Consulta_total_cod_uni.copy()
#-------------------------------------------------
Consulta_total_cod_uni=Consulta_total_cod_uni[Consulta_total_cod_uni['FuerzaComercialFinal'] != 'N/A']
Consulta_total_cod_uni=Consulta_total_cod_uni[Consulta_total_cod_uni['FuerzaComercialFinal'] != 'NA']
Consulta_total_cod_uni=Consulta_total_cod_uni[Consulta_total_cod_uni['FuerzaComercialFinal'] != 'PENDIENTE']
Consulta_total_cod_uni=Consulta_total_cod_uni[Consulta_total_cod_uni['FuerzaComercialFinal'].notna()]

#creo una consulta por codigo de unidad donde de verdad estan solo CONTACT
Consulta_recibos_per_COD_UNI_solo_CONTACT=Consulta_total_cod_uni[(Consulta_total_cod_uni['FuerzaComercialFinal'] == 'CONTACT')|(Consulta_total_cod_uni['FuerzaComercialFinal'] == 'CONTACT ')]

#AHORA SIMPLEMENTE COMPARO SI MIS LIQUIDACION ACTUAL EN SU RESPECTIVO PERIODO SI ES NUEVO
Etiquetas_penultimo_Con_cod_uni['Esta_en_consulta_CONTACT_CODUNI'] =Etiquetas_penultimo_Con_cod_uni['Llave_liq_cod_uni'].isin(Consulta_recibos_per_COD_UNI_solo_CONTACT['llave_total_cod_uni']).astype(int)
Etiquetas_penultimo_Con_cod_uni['Esta_en_consulta_total_fuerzas_CODUNI'] =Etiquetas_penultimo_Con_cod_uni['Llave_liq_cod_uni'].isin(Consulta_total_cod_uni['llave_total_cod_uni']).astype(int)

condicion_OTRA_fuerza_CODUNI=((Etiquetas_penultimo_Con_cod_uni['Esta_en_consulta_CONTACT_CODUNI']==0) & (Etiquetas_penultimo_Con_cod_uni['Esta_en_consulta_total_fuerzas_CODUNI']==1) &(~Etiquetas_penultimo_Con_cod_uni['TIPO'].str.contains('continuada', case=False, na=False)))
Etiquetas_penultimo_Con_cod_uni.loc[condicion_OTRA_fuerza_CODUNI, 'EstadoFinal'] = 'OTRA FUERZA DE VENTA_COD_U'
#--------------------------------------------------

# VALIDACION = NUEVO POR PRIMER PERIODO Y COD_UNI

In [22]:
Consulta_solo_para_validar_nuevo_por_uni.rename(columns={'PERIODO': 'Primer_periodo_real'}, inplace=True)

In [23]:
#De la consulta sin duplicados por llave de codigo de unidad. Me traigo el primer periodo real a las etiquetas. (Aqui no se generan duplicados
# en las etiquetas)

Etiquetas_penultimo_Con_cod_uni = Etiquetas_penultimo_Con_cod_uni.merge(Consulta_solo_para_validar_nuevo_por_uni[['llave_total_cod_uni', 'Primer_periodo_real']], 
                                                left_on='Llave_liq_cod_uni', 
                                                right_on='llave_total_cod_uni',
                                                how='left')
Etiquetas_penultimo_Con_cod_uni.drop(columns=['llave_total_cod_uni'], inplace=True)
Etiquetas_penultimo_Con_cod_uni.shape
#----------------------------------------------
#----------------------------------------------

(3477, 34)

In [24]:
#Si el periodo de la etiqueta no es igual al primer periodo real en el codigo de unidad =  ANTIGUO
condicion_antiguo_coduni=(
                (Etiquetas_penultimo_Con_cod_uni['Primer_periodo_real'].notna()) &
                (Etiquetas_penultimo_Con_cod_uni['PERIODO']!=Etiquetas_penultimo_Con_cod_uni['Primer_periodo_real']) &
                (~Etiquetas_penultimo_Con_cod_uni['TIPO'].str.contains('continuada', case=False, na=False)) 
)
Etiquetas_penultimo_Con_cod_uni.loc[condicion_antiguo_coduni, 'EstadoFinal'] = 'ANTIGUO_COD_U'

# Depuro columnas
Etiquetas_penultimo_Con_cod_uni.drop(columns=['Esta_en_consulta_Nuevos'], inplace=True)
Etiquetas_penultimo_Con_cod_uni.drop(columns=['Esta_en_consulta_per_Actual'], inplace=True)
Etiquetas_penultimo_Con_cod_uni.drop(columns=['Esta_en_consulta_CONTACT'], inplace=True)
Etiquetas_penultimo_Con_cod_uni.drop(columns=['Esta_en_consulta_per_Actual_total_fuerzas'], inplace=True)
Etiquetas_penultimo_Con_cod_uni.drop(columns=['Esta_en_consulta_CONTACT_CODUNI'], inplace=True)
Etiquetas_penultimo_Con_cod_uni.drop(columns=['Esta_en_consulta_total_fuerzas_CODUNI'], inplace=True)
Etiquetas_penultimo_Con_cod_uni.drop(columns=['Primer_periodo_real'], inplace=True)

Etiquetas_penultimo_Con_cod_uni.shape
#------------------------------------
#------------------------------------

(3477, 27)

In [25]:
#VUELVO A UNIR MI LIQUIDACION
Etiquetas_penultimo_paso=[]
Etiquetas_penultimo_paso = pd.concat([Etiquetas_penultimo_Con_cod_uni, Etiquetas_penultimo_paso_sin_cod_uni], ignore_index=True)
#--------------------------------
#--------------------------------

# CRUZAR POR REFERIDOS

In [26]:
host2_3 = '172.16.1.33'
port2_3 = '1433'
database2_3 = 'CUN_REPOSITORIO'

try:
    url2_3 = 'mssql+pyodbc://@{host}:{port}/{db}?trusted_connection=yes&driver=SQL+Server'.format(host=host2_3, port=port2_3, db=database2_3)
    engine2_3 = create_engine(url2_3) 
    print("Conexion a la base de datos realizada")
    Lista_ref = pd.read_sql("select * from CUN_REPOSITORIO.Comisiones.HISTOR_REFERIDOS", engine2_3)
    Lista_ref2 = Lista_ref.copy()
        
except Exception as e:
    print("Error:", e)
finally:
    # Cerrar la conexión
    if 'engine2_3' in locals():
        engine2_3.dispose()
        print("Conexión cerrada")
Lista_ref

Conexion a la base de datos realizada
Conexión cerrada


,Listado_Referidos,Ejecucion
0,1000731646,_2024-10-23_
1,1047369554,_2024-10-23_
2,80016486,_2024-10-23_
3,1064193348,_2024-10-23_
4,1001115701,_2024-10-23_
...,...,...
2117,79895868,_2025_08_02_
2118,80257124,_2025_08_02_
2119,80750452,_2025_08_02_
2120,8128046,_2025_08_02_


In [27]:
Lista_ref=Lista_ref.astype(str)
Etiquetas_penultimo_paso['Refer'] = Etiquetas_penultimo_paso['Doc_Estudiante'].isin(Lista_ref['Listado_Referidos']).astype(int)
Etiquetas_penultimo_paso.loc[Etiquetas_penultimo_paso['Refer'] == 1, 'EstadoFinal'] = 'OK'

# CRUZAR POR stv

In [28]:
host2_1 = '172.16.1.33'
port2_1 = '1433'
database2_1 = 'CUN_REPOSITORIO'

try:
    url2_1 = 'mssql+pyodbc://@{host}:{port}/{db}?trusted_connection=yes&driver=SQL+Server'.format(host=host2_1, port=port2_1, db=database2_1)
    engine2_1 = create_engine(url2_1) 
    print("Conexion a la base de datos realizada")
    cedulas_erroneas= pd.read_sql(f'''
select * from CUN_REPOSITORIO.JUAN_GUZMAN.CEDULAS_NO_LIQUIDAR
                            ''', engine2_1)
    cedulas_erroneas_copia =  cedulas_erroneas.copy()

    cedulas_erroneas3= pd.read_sql(f'''
    SELECT Documento_Estudiante_zoho, Periodo, FuerzaComercialFinal AS Fuerza_comercial
FROM CUN_REPOSITORIO.DBO.VENTAS_REINTEGROS
UNION ALL
SELECT Documento_Estudiante_zoho, Periodo, FuerzaComercialFinal
FROM CUN_REPOSITORIO.DBO.VENTAS_REINTEGROS_a
UNION ALL
SELECT Documento_Estudiante_zoho, Periodo, FuerzaComercialFinal
FROM CUN_REPOSITORIO.DBO.VENTAS_REINTEGROS_25V02
UNION ALL
SELECT Documento_Estudiante_zoho, Periodo, FuerzaComercialFinal
FROM CUN_REPOSITORIO.DBO.VENTAS_REINTEGROS_2025B
UNION ALL
SELECT Documento_Estudiante_zoho, Periodo, FuerzaComercialFinal
FROM CUN_REPOSITORIO.DBO.VENTAS_REINTEGROS_25V02_s
UNION ALL
SELECT Documento_Estudiante_zoho, Periodo, FuerzaComercialFinal
FROM CUN_REPOSITORIO.DBO.VENTAS_BASICA_MEDIA
UNION ALL
SELECT Documento_Estudiante_zoho, Periodo, FuerzaComercialFinal
FROM CUN_REPOSITORIO.DBO.VENTAS25V02_contact
UNION ALL
SELECT Documento_Estudiante_zoho, Periodo, FuerzaComercialFinal
FROM CUN_REPOSITORIO.DBO.VENTAS25V02_contact_2
UNION ALL
SELECT Documento_Estudiante_zoho, Periodo, FuerzaComercialFinal
FROM CUN_REPOSITORIO.DBO.VENTAS_REINTEGROS_25V02_2
UNION ALL
SELECT Documento_Estudiante_zoho, Periodo, FuerzaComercialFinal
FROM CUN_REPOSITORIO.DBO.VENTAS_REINTEGROS_2025B_2
UNION ALL
SELECT Documento_Estudiante_zoho, Periodo, FuerzaComercialFinal
FROM CUN_REPOSITORIO.DBO.VENTAS_REINTEGROS_25V02_3
UNION ALL
SELECT Documento_Estudiante_zoho, Periodo, FuerzaComercialFinal
FROM CUN_REPOSITORIO.DBO.VENTAS_REINTEGROS_25V02_4
UNION ALL
SELECT Documento_Estudiante_zoho, Periodo, FuerzaComercialFinal
FROM CUN_REPOSITORIO.DBO.VENTAS_REINTEGROS_2025B_3;
                            ''', engine2_1)
        
except Exception as e:
    print("Error:", e)
finally:
    # Cerrar la conexión
    if 'engine2_1' in locals():
        engine2_1.dispose()
        print("Conexión cerrada")
cedulas_erroneas['Cliente']=cedulas_erroneas['Documento_Estudiante_zoho'].astype(float).astype(int).astype(str)
'-----------------------------------------------------------------------------------------'
'-----------------------------------------------------------------------------------------'
cedulas_erroneas2 = pd.read_excel("EMORENO_Cedulas_reintegros_2025_ENERO.xlsx")
cedulas_erroneas2['Cliente']=cedulas_erroneas2['Cliente'].astype(float).astype(int).astype(str)
'-----------------------------------------------------------------------------------------'
'-----------------------------------------------------------------------------------------'
cedulas_erroneas = pd.concat([cedulas_erroneas, cedulas_erroneas2, cedulas_erroneas3], ignore_index=True)
cedulas_erroneas = cedulas_erroneas.drop_duplicates()


# Crear un set con las combinaciones válidas de cedulas y periodos
combos_invalidos = set(zip(cedulas_erroneas['Cliente'], cedulas_erroneas['PERIODO']))

# Aplicar validación combinada
Etiquetas_penultimo_paso['Reint_Stv'] = Etiquetas_penultimo_paso.apply(
    lambda row: int((row['Doc_Estudiante'], row['PERIODO']) in combos_invalidos), axis=1
)

Etiquetas_penultimo_paso.loc[Etiquetas_penultimo_paso['Reint_Stv'] == 1, 'EstadoFinal'] = 'OTRA_FUERZA_stv'

#---------------------------

Conexion a la base de datos realizada
Conexión cerrada


In [29]:
Etiquetas_penultimo_paso.shape

(3477, 33)

In [30]:
cedulas_erroneas_copia

,Documento_Estudiante_zoho,PERIODO
0,1000034074,2024B
1,1000049784,2024B
2,1000122716,2024B
3,1000124683,2024B
4,1000131678,2024B
...,...,...
5511,587347,25V02
5512,80007843,25V02
5513,80797755,25V02
5514,91531085,25V02


In [31]:
Etiquetas_penultimo_paso

,Doc_Estudiante,PERIODO,Documento_del_Asesor,Fecha_de_pago_Real,Propietario_del_registro,Programa_vendido,Fecha_de_pago,Correo_Asesor,Periodo_Pago,EST_MATRICULADO,...,Llave_liq_cod_uni,Esta_En_histor_telec_depur,Esta_En_histor_telec_NOved,Esta_En_histor_telec_TOTAL,Esta_en_consulta_Nuevos,Esta_en_consulta_per_Actual,Esta_en_consulta_CONTACT,Esta_en_consulta_per_Actual_total_fuerzas,Refer,Reint_Stv
0,1000003484,25V04,1014862000,14/07/2025,Contact Allison Campos Suarez,202507_INGENIERIA INDUSTRIAL,14/07/2025,ALLISON_CAMPOS@CUN.EDU.CO,202507.0,1-Activo,...,1000003484VTC09,0.0,0.0,NaN,NaN,NaN,NaN,NaN,0,0
1,1000005164,25V04,1003206797,28/07/2025,Contact Marjorie Villadiego Ballesteros,202507_INGENIERIA DE SISTEMAS,28/07/2025,MARJORIE_VILLADIEGO@CUN.EDU.CO,202507.0,1-Activo,...,1000005164VT15G,0.0,0.0,NaN,NaN,NaN,NaN,NaN,0,0
2,1000007318,25V04,1003518660,29/07/2025,Contact Jhon Eduardo Rodriguez Ortega,202507_INGENIERIA DE SISTEMAS,29/07/2025,JHON_RODRIGUEZ@CUN.EDU.CO,202507.0,1-Activo,...,1000007318VT15G,0.0,0.0,NaN,NaN,NaN,NaN,NaN,0,0
3,1000018721,25V04,1013589781,28/07/2025,Contact nicole jinneth hidalgo enciso,202507_INGENIERIA DE SISTEMAS,28/07/2025,NICOLE_HIDALGO@CUN.EDU.CO,202507.0,1-Activo,...,1000018721VT15G,0.0,0.0,NaN,NaN,NaN,NaN,NaN,0,0
4,1000031945,25V04,1085097684,28/07/2025,Contact Wilmar Giannini Loaiza Amado,202507_INGENIERIA DE SISTEMAS,28/07/2025,WILMAR_LOAIZA@CUN.EDU.CO,202507.0,1-Activo,...,1000031945VT15G,0.0,0.0,NaN,NaN,NaN,NaN,NaN,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3472,98629162,25V04,1003206797,11/07/2025,Contact Marjorie Villadiego Ballesteros,202507_ADMINISTRACION DE EMPRESAS AGROINDUSTRI...,11/07/2025,MARJORIE_VILLADIEGO@CUN.EDU.CO,202507.0,1-Activo,...,98629162VTC64,0.0,0.0,NaN,NaN,NaN,NaN,NaN,0,0
3473,98653204,25V04,1024595409,03/07/2025,Contact Aura Andrea Hipolito Garcia,202507_PUBLICIDAD Y MERCADEO,03/07/2025,AURA_HIPOLITO@CUN.EDU.CO,202507.0,1-Activo,...,98653204PVC08,0.0,0.0,NaN,NaN,NaN,NaN,NaN,0,0
3474,98715576,25V04,1085097684,19/07/2025,Contact Wilmar Giannini Loaiza Amado,202507_ADMINISTRACION PUBLICA,19/07/2025,WILMAR_LOAIZA@CUN.EDU.CO,202507.0,1-Activo,...,98715576TVM06,0.0,0.0,NaN,NaN,NaN,NaN,NaN,0,0
3475,98761179,25V04,1023030467,28/07/2025,Contact YEIMI LORENA BUSTOS BOSA,202507_INGENIERIA DE SISTEMAS,28/07/2025,YEIMI_BUSTOSB@CUN.EDU.CO,202507.0,1-Activo,...,98761179VT15G,0.0,0.0,NaN,NaN,NaN,NaN,NaN,0,0


In [32]:
df_filtrado_1=Etiquetas_penultimo_paso.copy()

# VALIDACION - YA SE PAGO - MISMO CONTACT

In [33]:
host2_2 = '172.16.1.33'
port2_2 = '1433'
database2_2= 'CUN_REPOSITORIO'

try:
    url2_2 = 'mssql+pyodbc://@{host}:{port}/{db}?trusted_connection=yes&driver=SQL+Server'.format(host=host2_2, port=port2_2, db=database2_2)
    engine2_2= create_engine(url2_2) 
    print("Conexion a la base de datos realizada")
    archivo_contact = pd.read_sql(f'''
            select * from CUN_REPOSITORIO.comisiones_contact.etiquetas 
                    where EstadoFinal='OK' and Periodo_Pago <>{MES_ACTUAL_EN_LIQUIDACION}
                                  ''', engine2_2)
    archivo_contact2 = archivo_contact.copy()
        
except Exception as e:
    print("Error:", e)
finally:
    # Cerrar la conexión
    if 'engine2_2' in locals():
        engine2_2.dispose()
        print("Conexión cerrada")
archivo_contact.shape

Conexion a la base de datos realizada
Conexión cerrada


(30692, 20)

In [34]:
#VALIDACION QUE SI FUNCIONO DEFINIR EL MES ARRIBA
# archivo_contact[archivo_contact['Periodo_Pago']==202501]

In [35]:
host2_4 = '172.16.1.33'
port2_4 = '1433'
database2_4= 'CUN_REPOSITORIO'

try:
    url2_4 = 'mssql+pyodbc://@{host}:{port}/{db}?trusted_connection=yes&driver=SQL+Server'.format(host=host2_4, port=port2_4, db=database2_4)
    engine2_4= create_engine(url2_4) 
    print("Conexion a la base de datos realizada")
    archivo_contact3 = pd.read_sql("select * from CUN_REPOSITORIO.comisiones_contact.EtiquetasReliquidacion where EstadoFinal='OK'", engine2_4)
    archivo_contact4 = archivo_contact3.copy()
        
except Exception as e:
    print("Error:", e)
finally:
    # Cerrar la conexión
    if 'engine2_4' in locals():
        engine2_4.dispose()
        print("Conexión cerrada")

Conexion a la base de datos realizada
Conexión cerrada


In [36]:
host2_4 = '172.16.1.33'
port2_4 = '1433'
database2_4= 'CUN_REPOSITORIO'

try:
    url2_4 = 'mssql+pyodbc://@{host}:{port}/{db}?trusted_connection=yes&driver=SQL+Server'.format(host=host2_4, port=port2_4, db=database2_4)
    engine2_4= create_engine(url2_4) 
    print("Conexion a la base de datos realizada")
    CONTACT_UNICD = pd.read_sql('''
select * from CUN_REPOSITORIO.comisiones_CONTACT.CONTACT_UNICD
WHERE COD_UNI IS NOT NULL
                                ''', engine2_4)
        
except Exception as e:
    print("Error:", e)
finally:
    # Cerrar la conexión
    if 'engine2_4' in locals():
        engine2_4.dispose()
        print("Conexión cerrada")

Conexion a la base de datos realizada
Conexión cerrada


ACA YA TENGO TODAS LAS ETIQUETAS OK (PAGADAS) - DE VENTAS NORMAL Y RELIQUIDACION

In [37]:
archivo_contact2 = pd.concat([archivo_contact2, archivo_contact3], ignore_index=True)
archivo_contact2['Doc_Estudiante']=archivo_contact2['Doc_Estudiante'].astype(str)

Solo me quedo de esas etiquetas OK solo las cedulas que estan en la liquidacion y me traigo el codigo unidad a esas etiquetas contact

In [38]:
archivo_contact2=archivo_contact2[archivo_contact2['Doc_Estudiante'].isin(df_filtrado_1['Doc_Estudiante'])]
archivo_contact2['Llave_etiq_cont']=archivo_contact2['Doc_Estudiante']+archivo_contact2['PERIODO']
# Quito duplicados de CONTACT_UNICD para añadir el cod_uni a las etiquetas
CONTACT_UNICD = CONTACT_UNICD.drop_duplicates(subset='llave_per', keep='first')

archivo_contact2 = archivo_contact2.merge(CONTACT_UNICD[['llave_per', 'COD_UNI']], 
                                     left_on='Llave_etiq_cont', 
                                     right_on='llave_per', 
                                     how='left')
archivo_contact2.rename(columns={'COD_UNI': 'COD_UNI_CONTACT'}, inplace=True)
archivo_contact2.drop(columns=['llave_per'], inplace=True)
#-----------------------------------------------
#-----------------------------------------------

VALIDACION VS HISTORICO CONTACT DIRECTO POR PERIODO

In [39]:
df_filtrado_1['Esta_en_hist_Cont_x_per']=df_filtrado_1['Llave_per'].isin(archivo_contact2['Llave_etiq_cont']).astype(int)
df_filtrado_1.loc[df_filtrado_1['Esta_en_hist_Cont_x_per']==1, 'EstadoFinal'] = 'YA SE PAGO'

VALIDACION VS HISTORICO CONCTACT POR CODIGO DE UNIDAD

In [40]:
"Nuevamente Divido mi liquidacion entre los que si obtuvieron su cod_uni y los que no"

df_final_sin_uni=df_filtrado_1[df_filtrado_1['Llave_liq_cod_uni'].isna()]
df_filtrado_1=df_filtrado_1[df_filtrado_1['Llave_liq_cod_uni'].notna()]

# ACA VALIDO PARA LOS QUE EN MI LIQ ACTUAL SI TIENEN CODIGO DE UNIDAD

Primero valido de mi liquidcion los que si tienen codigo de uidsad con una funcion que contemple dos cosas
- Los que en las etiquets si tienen cod uni
- Los que en las etiquetas no tiene cod uni

In [41]:
def comparar_y_agregar_pert_contact(df_final, archivo_contact2):
    # Iterar sobre cada fila de df_final
    for idx, row in df_final.iterrows():
        # Obtener el valor de 'DOC_ALUM', 'COD_UNI' y 'EstadoFinal' de la fila actual de df_final
        doc_alum = row['Doc_Estudiante']
        cod_uni = row['COD_UNI']
        estado_fini = row['EstadoFinal']
        
        # Buscar en archivo_contact2 si existe alguna fila que tenga 'Doc_Estudiante' igual a 'DOC_ALUM'
        match = archivo_contact2[archivo_contact2['Doc_Estudiante'] == doc_alum]
        
        # Si no se encuentra coincidencia en 'Doc_Estudiante'
        if match.empty:
            df_final.at[idx, 'EstadoFinal'] = estado_fini
        else:
            # Si se encuentra coincidencia, comprobar la columna 'COD_UNI_CONTACT'
            cod_uni_contact = match['COD_UNI_CONTACT'].values[0]
            
            if pd.isna(cod_uni_contact):  # Si 'COD_UNI_CONTACT' es NaN
                df_final.at[idx, 'EstadoFinal'] = 'YA SE PAGO'
            elif cod_uni == cod_uni_contact:  # Si 'COD_UNI' coincide con 'COD_UNI_CONTACT'
                df_final.at[idx, 'EstadoFinal'] = 'YA SE PAGO'
            else:  # Si no coincide
                df_final.at[idx, 'EstadoFinal'] = estado_fini
    
    return df_final

In [42]:
df_filtrado_1 = comparar_y_agregar_pert_contact(df_filtrado_1, archivo_contact2)

# ACA VALIDO PARA LOS QUE EN MI LIQ ACTUAL NO TIENEN CODIGO DE UNIDAD

In [43]:
df_final_sin_uni['Esta_en_hist_cont'] = df_final_sin_uni['Doc_Estudiante'].isin(archivo_contact2['Doc_Estudiante']).astype(int)
df_final_sin_uni.loc[df_final_sin_uni['Esta_en_hist_cont'] == 1, 'EstadoFinal'] = 'YA SE PAGO'

Uno nuevamente mi liquidacion que habia dividido en dos partes

In [44]:
df_filtrado_1 = pd.concat([df_filtrado_1, df_final_sin_uni], ignore_index=True)

Preparo la liq actual para inyectarla en CONTAC_UNICD

In [45]:
df_filtrado_1.rename(columns={'Llave_per': 'llave_per'}, inplace=True)
CONTAC_UNI_TEMP=df_filtrado_1[['Doc_Estudiante',	'PERIODO',	'Documento_del_Asesor',	'Fecha_de_pago_Real',	'Propietario_del_registro',	
                             'Programa_vendido',	'Fecha_de_pago',	'Correo_Asesor',	'Periodo_Pago',	'EST_MATRICULADO',	
                             'NCR_ANULACION',	'TIPO',	'ORDEN',	'VAL_LIQUIDADO',	'NUEVO',	'ESTADO_PAGO',	'Documento_Estudiante_zoho',
                             'Motivo',	'Propietario_de_Fase_Name',	'EstadoFinal',	'llave_per',	'COD_UNI']]

In [46]:
host_2_12 = '172.16.1.33'
port_2_12 = '1433'
database_2_12 = 'CUN_REPOSITORIO'
url_2_12 = 'mssql+pyodbc://@{host}:{port}/{db}?driver=ODBC Driver 17 for SQL Server'.format(host=host_2_12, port=port_2_12, db=database_2_12)
engine_2_12 = create_engine(url_2_12, fast_executemany=True)

table_name_2_12 = 'CONTACT_UNICD'
schema_name_2_12 = 'COMISIONES_CONTACT'
# # Send the data to sql server
CONTAC_UNI_TEMP.to_sql(
      name=table_name_2_12,
      schema=schema_name_2_12,
      con=engine_2_12,
      if_exists='append',
      index=False,
#
     chunksize = 10000
  )

# # End connection
engine_2_12.dispose()

In [47]:
#VALIDADO - SI TIENEN LAS MISMAS COLUMNAS PARA INYECCION, MISMO ORDEN Y MISMO TIPO

# dtypes = df_filtrado_1.dtypes
# dtypes_df = pd.DataFrame(dtypes).reset_index()
# dtypes_df.columns = ['Columna', 'Tipo de Dato']
# dtypes_df#.to_excel('tipos_de_dato_final_base_pago.xlsx', index=False)
# # Tabla_Conteos_SP.dtypes

Preparacion para inyectar en TEMP_ETIQ_ACTUAL_1_1

In [48]:
df_filtrado_1=df_filtrado_1[['Doc_Estudiante',	'PERIODO',	'Documento_del_Asesor',	'Fecha_de_pago_Real',	'Propietario_del_registro',
                             'Programa_vendido',	'Fecha_de_pago',	'Correo_Asesor',	'Periodo_Pago',	'EST_MATRICULADO',	'NCR_ANULACION',
                             'TIPO',	'ORDEN',	'VAL_LIQUIDADO',	'NUEVO',	'ESTADO_PAGO',	'Documento_Estudiante_zoho',	'Motivo',	
                             'Propietario_de_Fase_Name',	'EstadoFinal']]

In [49]:
df_filtrado_1

,Doc_Estudiante,PERIODO,Documento_del_Asesor,Fecha_de_pago_Real,Propietario_del_registro,Programa_vendido,Fecha_de_pago,Correo_Asesor,Periodo_Pago,EST_MATRICULADO,NCR_ANULACION,TIPO,ORDEN,VAL_LIQUIDADO,NUEVO,ESTADO_PAGO,Documento_Estudiante_zoho,Motivo,Propietario_de_Fase_Name,EstadoFinal
0,1000003484,25V04,1014862000,14/07/2025,Contact Allison Campos Suarez,202507_INGENIERIA INDUSTRIAL,14/07/2025,ALLISON_CAMPOS@CUN.EDU.CO,202507.0,1-Activo,0.0,Pregrado,3088952.0,3653500.0,NUEVO,PAGO,1000003484,ok,allison_campos@cun.edu.co,OK
1,1000005164,25V04,1003206797,28/07/2025,Contact Marjorie Villadiego Ballesteros,202507_INGENIERIA DE SISTEMAS,28/07/2025,MARJORIE_VILLADIEGO@CUN.EDU.CO,202507.0,1-Activo,0.0,Pregrado,3109384.0,3653500.0,NUEVO,PAGO,1000005164,ok,Marjorie_villadiego@cun.edu.co,OK
2,1000007318,25V04,1003518660,29/07/2025,Contact Jhon Eduardo Rodriguez Ortega,202507_INGENIERIA DE SISTEMAS,29/07/2025,JHON_RODRIGUEZ@CUN.EDU.CO,202507.0,1-Activo,0.0,Pregrado,3109884.0,3653500.0,NUEVO,PAGO,1000007318,ok,jhon_rodriguez@cun.edu.co,OK
3,1000018721,25V04,1013589781,28/07/2025,Contact nicole jinneth hidalgo enciso,202507_INGENIERIA DE SISTEMAS,28/07/2025,NICOLE_HIDALGO@CUN.EDU.CO,202507.0,1-Activo,0.0,Pregrado,3111799.0,3653500.0,NUEVO,PAGO,1000018721,ok,nicole_hidalgo@cun.edu.co,OK
4,1000031945,25V04,1085097684,28/07/2025,Contact Wilmar Giannini Loaiza Amado,202507_INGENIERIA DE SISTEMAS,28/07/2025,WILMAR_LOAIZA@CUN.EDU.CO,202507.0,1-Activo,0.0,Pregrado,3108348.0,3653500.0,NUEVO,PAGO,1000031945,ok,wilmar_loaiza@cun.edu.co,OK
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3472,98629162,25V04,1003206797,11/07/2025,Contact Marjorie Villadiego Ballesteros,202507_ADMINISTRACION DE EMPRESAS AGROINDUSTRI...,11/07/2025,MARJORIE_VILLADIEGO@CUN.EDU.CO,202507.0,1-Activo,0.0,Pregrado,3091355.0,3480600.0,NUEVO,PAGO,98629162,ok,Marjorie_villadiego@cun.edu.co,OK
3473,98653204,25V04,1024595409,03/07/2025,Contact Aura Andrea Hipolito Garcia,202507_PUBLICIDAD Y MERCADEO,03/07/2025,AURA_HIPOLITO@CUN.EDU.CO,202507.0,1-Activo,0.0,Pregrado,3067486.0,3480600.0,NUEVO,PAGO,98653204,ok,aura_hipolito@cun.edu.co,OK
3474,98715576,25V04,1085097684,19/07/2025,Contact Wilmar Giannini Loaiza Amado,202507_ADMINISTRACION PUBLICA,19/07/2025,WILMAR_LOAIZA@CUN.EDU.CO,202507.0,1-Activo,0.0,Pregrado,3102085.0,3480600.0,NUEVO,PAGO,98715576,ok,wilmar_loaiza@cun.edu.co,OK
3475,98761179,25V04,1023030467,28/07/2025,Contact YEIMI LORENA BUSTOS BOSA,202507_INGENIERIA DE SISTEMAS,28/07/2025,YEIMI_BUSTOSB@CUN.EDU.CO,202507.0,1-Activo,0.0,Pregrado,3111516.0,3653500.0,NUEVO,PAGO,98761179,ok,yeimi_bustosb@cun.edu.co,OK


In [50]:
host_2_12 = '172.16.1.33'
port_2_12 = '1433'
database_2_12 = 'CUN_REPOSITORIO'
url_2_12 = 'mssql+pyodbc://@{host}:{port}/{db}?driver=ODBC Driver 17 for SQL Server'.format(host=host_2_12, port=port_2_12, db=database_2_12)
engine_2_12 = create_engine(url_2_12, fast_executemany=True)

table_name_2_12 = 'ETIQUETAS'
schema_name_2_12 = 'COMISIONES_CONTACT'
# # Send the data to sql server
df_filtrado_1.to_sql(
     name=table_name_2_12,
     schema=schema_name_2_12,
     con=engine_2_12,
     if_exists='append',
     index=False,
     chunksize = 10000
 )

# # End connection
engine_2_12.dispose()

C:\Users\alejandro_arboledas\AppData\Local\Temp\ipykernel_5868\1462192386.py:10: UserWarning: The provided table name 'ETIQUETAS' is not found exactly as such in the database after writing the table, possibly due to case sensitivity issues. Consider using lower case table names.
  df_filtrado_1.to_sql(


In [51]:
import datetime
import calendar

# Obtener la fecha actual
hoy = datetime.date.today()

# Calcular el primer día del mes actual y restar un día para obtener el mes anterior
primer_dia_mes_actual = hoy.replace(day=1)
ultimo_dia_mes_anterior = primer_dia_mes_actual - datetime.timedelta(days=1)

# Extraer mes y año del mes anterior
mes_anterior = ultimo_dia_mes_anterior.month
anio_anterior = ultimo_dia_mes_anterior.year

# Asignar variables
MES_SQL_SP_LIQ_ACTUAL = f"'{anio_anterior}{mes_anterior:02d}'"
PERIODO_ALMACEN = f"{anio_anterior}{mes_anterior:02d}"
PERIODO_NORMAL_float = f"{anio_anterior}{mes_anterior:02d}"

# Imprimir resultados
print("SP =", MES_SQL_SP_LIQ_ACTUAL)
print("PERIODO_FINANCIERO =", PERIODO_ALMACEN)
print("PERIODO_NORMAL_float =", PERIODO_NORMAL_float)

SP = '202507'
PERIODO_FINANCIERO = 202507
PERIODO_NORMAL_float = 202507


In [52]:
# Configura los detalles de la conexión
host='172.16.1.33'
port='1433'
database= 'CUN_REPOSITORIO'

try:
    # Crea la cadena de conexión
   url = 'mssql+pyodbc://@{host}:{port}/{db}?driver=SQL+Server'.format(host=host, port=port, db=database)

   
    # Crear conexion con base de datos
   engine = create_engine(url) 
   print("Conexion a la base de datos realizada") 
    
except Exception as e:
    print("Error:", e)

Conexion a la base de datos realizada


In [53]:
#CONSULTA PARA CREAR LA TABLA TEMPORAL EN SQL 
with engine.connect() as connection:
    try:

        sql_query = text(f'''
         IF object_id('CUN_REPOSITORIO.COMISIONES_CONTACT.TEMP_CONTEO_VENTAS') IS NOT NULL
            DROP TABLE CUN_REPOSITORIO.COMISIONES_CONTACT.TEMP_CONTEO_VENTAS


        Delete from Comisiones_Contact.FIJA_LIQ_ACTUAL_DESGLOS where Periodo_Pago = {MES_SQL_SP_LIQ_ACTUAL}
        Delete from Comisiones_Contact.PREGR_APROB_CONTAC_MOD2 where Periodo_Pago_1 = {MES_SQL_SP_LIQ_ACTUAL}
        Delete from Comisiones_Contact.POSG_APROB_CONTAC_MOD2 where Periodo_Pago_1 = {MES_SQL_SP_LIQ_ACTUAL}
        ''')
        connection.execute(sql_query)
        connection.commit()  # Asegurar que los cambios se guarden en la base de datos
        print("Consulta realizada con exito")
    except Exception as e:
        print(f"Error al eliminar registros: {e}")
    finally:
        connection.close()

Consulta realizada con exito


In [54]:
#ACA CONSULTO SOLO MIS ETIQUETAS OK PARA EXCLUSIVAMENTE SACAR LA TABLA DE CONTEOS
#-------------------------------------------------------------------------------
host2_8 = '172.16.1.33'
port2_8 = '1433'
database2_8= 'CUN_REPOSITORIO'

try:
    url2_8 = 'mssql+pyodbc://@{host}:{port}/{db}?trusted_connection=yes&driver=SQL+Server'.format(host=host2_8, port=port2_8, db=database2_8)
    engine2_8= create_engine(url2_8) 
    print("Conexion a la base de datos realizada")
    Etiquetas_mes_acutual = pd.read_sql(f'''
select * from COMISIONES_CONTACT.ETIQUETAS where Periodo_Pago ={MES_SQL_SP_LIQ_ACTUAL} and EstadoFinal = 'OK'
                            ''', engine2_8)
    Etiquetas_mes_acutual2 = Etiquetas_mes_acutual.copy()
        
except Exception as e:
    print("Error:", e)
finally:
    if 'engine2_8' in locals():
        engine2_8.dispose()
        print("Conexión cerrada")
Etiquetas_mes_acutual['Llave_etiq_per']=Etiquetas_mes_acutual['Doc_Estudiante'].astype(str)+Etiquetas_mes_acutual['PERIODO'].astype(str)
Etiquetas_mes_acutual.to_excel('etiqutas_iniciales_orig.xlsx')
Etiquetas_mes_acutual

Conexion a la base de datos realizada
Conexión cerrada


,Doc_Estudiante,PERIODO,Documento_del_Asesor,Fecha_de_pago_Real,Propietario_del_registro,Programa_vendido,Fecha_de_pago,Correo_Asesor,Periodo_Pago,EST_MATRICULADO,...,TIPO,ORDEN,VAL_LIQUIDADO,NUEVO,ESTADO_PAGO,Documento_Estudiante_zoho,Motivo,Propietario_de_Fase_Name,EstadoFinal,Llave_etiq_per
0,1000929330,2025C,1000354487,31/07/2025,Contact Cristian Javier Garcia Barreto,202507_DISEO DE MODAS,31/07/2025,CRISTIAN_GARCIAB@CUN.EDU.CO,202507.0,1-Activo,...,Pregrado,3116398.0,2816600.0,NUEVO,PAGO,1000929330,ok,cristian_garciab@cun.edu.co,OK,10009293302025C
1,1000937029,25V04,1012454417,30/07/2025,Contact Elizabeth Villarreal Sanchez,202507_ADMINISTRACIN DE EMPRESAS,30/07/2025,ELIZABETH_VILLARREAL@CUN.EDU.CO,202507.0,1-Activo,...,Pregrado,3116155.0,5139800.0,NUEVO,PAGO,1000937029,ok,elizabeth_villarreal@cun.edu.co,OK,100093702925V04
2,1000940668,2025C,1019095506,29/07/2025,Contact Brayan Sting Cespedes Piza,202507_INGENIERIA DE SISTEMAS,29/07/2025,BRAYAN_CESPEDESP@CUN.EDU.CO,202507.0,1-Activo,...,Pregrado,3138406.0,2816600.0,NUEVO,PAGO,1000940668,ok,brayan_cespedesp@cun.edu.co,OK,10009406682025C
3,1000940925,25V04,1014862000,22/07/2025,Contact Allison Campos Suarez,202507_DISEO GRAFICO,22/07/2025,ALLISON_CAMPOS@CUN.EDU.CO,202507.0,1-Activo,...,Pregrado,3105271.0,3653500.0,NUEVO,PAGO,1000940925,ok,allison_campos@cun.edu.co,OK,100094092525V04
4,1000944533,25V04,52498031,09/07/2025,Contact BERTHA ISABEL RAMOS SASTOQUE,202507_ADMINISTRACIN DE EMPRESAS,09/07/2025,BERTHA_RAMOS@CUN.EDU.CO,202507.0,1-Activo,...,Pregrado,3089925.0,5139800.0,NUEVO,PAGO,1000944533,ok,bertha_ramos@cun.edu.co,OK,100094453325V04
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3408,98629162,25V04,1003206797,11/07/2025,Contact Marjorie Villadiego Ballesteros,202507_ADMINISTRACION DE EMPRESAS AGROINDUSTRI...,11/07/2025,MARJORIE_VILLADIEGO@CUN.EDU.CO,202507.0,1-Activo,...,Pregrado,3091355.0,3480600.0,NUEVO,PAGO,98629162,ok,Marjorie_villadiego@cun.edu.co,OK,9862916225V04
3409,98653204,25V04,1024595409,03/07/2025,Contact Aura Andrea Hipolito Garcia,202507_PUBLICIDAD Y MERCADEO,03/07/2025,AURA_HIPOLITO@CUN.EDU.CO,202507.0,1-Activo,...,Pregrado,3067486.0,3480600.0,NUEVO,PAGO,98653204,ok,aura_hipolito@cun.edu.co,OK,9865320425V04
3410,98715576,25V04,1085097684,19/07/2025,Contact Wilmar Giannini Loaiza Amado,202507_ADMINISTRACION PUBLICA,19/07/2025,WILMAR_LOAIZA@CUN.EDU.CO,202507.0,1-Activo,...,Pregrado,3102085.0,3480600.0,NUEVO,PAGO,98715576,ok,wilmar_loaiza@cun.edu.co,OK,9871557625V04
3411,98761179,25V04,1023030467,28/07/2025,Contact YEIMI LORENA BUSTOS BOSA,202507_INGENIERIA DE SISTEMAS,28/07/2025,YEIMI_BUSTOSB@CUN.EDU.CO,202507.0,1-Activo,...,Pregrado,3111516.0,3653500.0,NUEVO,PAGO,98761179,ok,yeimi_bustosb@cun.edu.co,OK,9876117925V04


In [55]:
#EN ESTOS PASOS TOMO LAS CEDULAS UNICAS DE ESTUDIANTE PARA FILTRARLAS EN LOS RECIBOS DE CAJA - ME TOCO DIVIDIRLAS
#SI EN TOTAL SON 2500 CEDULA,  YA PROBE Y SI EN cedulas_1 hubiesen 500 cedulas si funciona
#------------------------------------------------------------------------------------------
import numpy as np
Cedulas_filtro = Etiquetas_mes_acutual['Doc_Estudiante'].unique()
num_partes = 10
size = len(Cedulas_filtro) // num_partes
residual = len(Cedulas_filtro) % num_partes
cedulas_partes = []
start = 0
for i in range(num_partes):
    end = start + size + (1 if i < residual else 0)
    cedulas_partes.append(Cedulas_filtro[start:end])
    start = end
cedulas_1, cedulas_2, cedulas_3, cedulas_4, cedulas_5, cedulas_5_1, cedulas_7, cedulas_8, cedulas_9, cedulas_10 = cedulas_partes
#----------------------------------------------------------------------------------
Cedulas_filtro1 = cedulas_1
cedulas_consulta1 = ", ".join([f"''{cedula1}''" for cedula1 in Cedulas_filtro1])
Cedulas_filtro2 = cedulas_2
cedulas_consulta2 = ", ".join([f"''{cedula2}''" for cedula2 in Cedulas_filtro2])
Cedulas_filtro3 = cedulas_3
cedulas_consulta3= ", ".join([f"''{cedula3}''" for cedula3 in Cedulas_filtro3])
Cedulas_filtro4 = cedulas_4
cedulas_consulta4= ", ".join([f"''{cedula4}''" for cedula4 in Cedulas_filtro4])
Cedulas_filtro5 = cedulas_5
cedulas_consulta5= ", ".join([f"''{cedula5}''" for cedula5 in Cedulas_filtro5])
Cedulas_filtro5_1 = cedulas_5_1
cedulas_consulta5_1= ", ".join([f"''{cedula5_1}''" for cedula5_1 in Cedulas_filtro5_1])
Cedulas_filtro7 = cedulas_7
cedulas_consulta7= ", ".join([f"''{cedula7}''" for cedula7 in Cedulas_filtro7])
Cedulas_filtro8 = cedulas_8
cedulas_consulta8= ", ".join([f"''{cedula8}''" for cedula8 in Cedulas_filtro8])
Cedulas_filtro9 = cedulas_9
cedulas_consulta9= ", ".join([f"''{cedula9}''" for cedula9 in Cedulas_filtro9])
Cedulas_filtro10 = cedulas_10
cedulas_consulta10= ", ".join([f"''{cedula10}''" for cedula10 in Cedulas_filtro10])
#------------------------------------------------------------------------------------
cedulas_dict = {1: cedulas_consulta1,2: cedulas_consulta2,3: cedulas_consulta3,4: cedulas_consulta4,5: cedulas_consulta5,6: cedulas_consulta5_1,7: cedulas_consulta7,8: cedulas_consulta8,9: cedulas_consulta9,10: cedulas_consulta10}



In [56]:
#ACA FILTRO LAS CEDULAS DE ESTUDIANTE EN LOS RECIBOS DE CAJA (TOCO CON UN FOR) YA ESTA FUNCIONAL
#-------------------------------------------------------------------------------
import time
Recibos_Caja = pd.DataFrame()
for i in range(1, 11):
    cedulas_consulta = cedulas_dict[i]
    host2_9 = '172.16.1.33'
    port2_9 = '1433'
    database2_9= 'CUN_REPOSITORIO'
    try:
        url2_9 = 'mssql+pyodbc://@{host}:{port}/{db}?trusted_connection=yes&driver=SQL+Server'.format(host=host2_9, port=port2_9, db=database2_9)
        engine2_9= create_engine(url2_9) 
        print("Conexion a la base de datos realizada")
        df = pd.read_sql(f'''
            SELECT
                *
            FROM OPENQUERY([172.16.1.175],
            'SELECT * FROM iceberg.recibo_caja WHERE CLIENTE IN ({cedulas_consulta}) AND ESTADO = ''V'';')
                                ''', engine2_9)
        Recibos_Caja = pd.concat([Recibos_Caja, df], ignore_index=True)
    except Exception as e:
        print("Error:", e)
    finally:
        if 'engine2_9' in locals():
            engine2_9.dispose()
            print("Conexión cerrada")
    time.sleep(2)
Recibos_Caja

Conexion a la base de datos realizada
Conexión cerrada
Conexion a la base de datos realizada
Conexión cerrada
Conexion a la base de datos realizada
Conexión cerrada
Conexion a la base de datos realizada
Conexión cerrada
Conexion a la base de datos realizada
Conexión cerrada
Conexion a la base de datos realizada
Conexión cerrada
Conexion a la base de datos realizada
Conexión cerrada
Conexion a la base de datos realizada
Conexión cerrada
Conexion a la base de datos realizada
Conexión cerrada
Conexion a la base de datos realizada
Conexión cerrada


,ORGANIZACION,NUMERO,CAJA,SECUENCIA_PERSONA,FECHA,DESCRIPCION,VALOR,ESTADO,CLIENTE,FECHA_PAGO,PERIODO,CONSECUTIVO
0,1,28514451.0,301,208546.0,2015-08-29 00:00:00.0000000,MATRICULA 2015-B. APLICACION AUTOMATICA DE REC...,0.0,V,10781139.0,2015-08-28 00:00:00.0000000,2015B,NaN
1,1,30307189.0,100,208546.0,2015-12-06 00:00:00.0000000,Cruce AUTOMATICO DE SALDOS hecho al dia 01-DEC-15,0.0,V,10781139.0,2015-12-31 00:00:00.0000000,2015B,NaN
2,1,29126769.0,100,208546.0,2015-10-06 00:00:00.0000000,Cruce AUTOMATICO DE SALDOS hecho al dia 06-OCT-15,0.0,V,10781139.0,2015-10-31 00:00:00.0000000,2015B,NaN
3,1,29674861.0,100,208546.0,2015-11-05 00:00:00.0000000,Cruce AUTOMATICO DE SALDOS hecho al dia 30-OCT-15,0.0,V,10781139.0,2015-11-30 00:00:00.0000000,2015B,NaN
4,1,29126508.0,301,208546.0,2015-10-06 00:00:00.0000000,ABON DEUDA.,0.0,V,10781139.0,2015-10-02 00:00:00.0000000,2015B,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
14515,1,280856448.0,201,496868.0,2025-07-09 10:25:33.0000000,Pago automatico por Boton de Pagos. Referencia...,1164590.0,V,98778839.0,2025-07-09 10:25:33.0000000,25V04,2223441.0
14516,1,283207719.0,201,500771.0,2025-07-18 11:02:01.0000000,Pago automatico por Boton de Pagos. Referencia...,140087.0,V,536561824.0,2025-07-18 11:02:01.0000000,25V04,2232475.0
14517,1,283207640.0,103,500771.0,2025-07-18 00:00:00.0000000,Recibo creado AUTOMATICAMENTE en la Activación...,0.0,V,536561824.0,2025-07-18 00:00:00.0000000,25V04,NaN
14518,1,283207641.0,103,500771.0,2025-07-18 00:00:00.0000000,Cruce AUTOMATICO DE SALDOS -VIRTUAL- hecho has...,0.0,V,536561824.0,2025-07-18 00:00:00.0000000,25V04,NaN


In [57]:
Recibos_Caja['CLIENTE'] = Recibos_Caja['CLIENTE'].fillna(0).astype(int).astype(str)
#aca filtro el dataframe solo con los pagos que corresponden a los mismo periodos que en etiquetas
Recibos_Caja['Llave_per_caja']=Recibos_Caja['CLIENTE'].astype(str)+Recibos_Caja['PERIODO'].astype(str)
Recibos_Caja = Recibos_Caja[Recibos_Caja['Llave_per_caja'].isin(Etiquetas_mes_acutual['Llave_etiq_per'])]
Recibos_Caja = Recibos_Caja.drop('Llave_per_caja', axis=1)
#---------------------------------------------------------------------------
#ACA AÑADO LAS ETIQUETAS EL VALOR PAGADO POR EL ESTUDIANTE
#---------------------------------------------------------------------------
Recibos_Caja_t=Recibos_Caja[['CLIENTE','VALOR']]
df_resumen_recibos = Recibos_Caja_t.groupby('CLIENTE', as_index=False)['VALOR'].sum()

In [58]:
#AQUÍ FORMO LA TABLA DE LO QUE TEORICAMENTE DEBIA PAGAR EL ESTUDIANTE 
#------------------------------------------------------------------------------------------------------------
Cedulas_filtroz = Etiquetas_mes_acutual['Doc_Estudiante'].unique()
cedulas_para_zoho = ", ".join([f"'{cedulaz}'" for cedulaz in Cedulas_filtroz])
#------------------------------------------------------------------------------------------------------------
host2_1 = '172.16.1.33'
port2_1 = '1433'
database2_1 = 'CUN_REPOSITORIO'
try:
    url2_1 = 'mssql+pyodbc://@{host}:{port}/{db}?trusted_connection=yes&driver=SQL+Server'.format(host=host2_1, port=port2_1, db=database2_1)
    engine2_1 = create_engine(url2_1) 
    print("Conexion a la base de datos realizada")
    ventas_zoho = pd.read_sql(f'''
SELECT DISTINCT
    Documento_Estudiante_zoho,
	PERIODO,
	FEC_PAGO_LIQ,
    VALOR_ORDEN,
    val_becdtos,
    valor_ingles,
    ADICIONALES
FROM CUN_REPOSITORIO.DBO.Recibos_Contact2_Sin_Zoho with (nolock)
WHERE
   (ESTADO_PAGO = 'PAGO'
   AND EST_MATRICULADO = '1-Activo'
   AND DOCUMENTO IN ('FAMA','FECO')
   and PERIODO not like '%I%')
   AND Documento_Estudiante_zoho IN ({cedulas_para_zoho})
                               ''', engine2_1)
    ventas_zoho2 = ventas_zoho.copy()
except Exception as e:
    print("Error:", e)
finally:
    if 'engine2_1' in locals():
        engine2_1.dispose()
        print("Conexión cerrada")
ventas_zoho

Conexion a la base de datos realizada
Conexión cerrada


,Documento_Estudiante_zoho,PERIODO,FEC_PAGO_LIQ,VALOR_ORDEN,val_becdtos,valor_ingles,ADICIONALES
0,1000003484,25V04,13/07/2025,3653500.0,-2929750.0,200000.0,23000.0
1,1000005164,25V04,26/07/2025,3653500.0,-2929750.0,200000.0,23000.0
2,1000007318,25V04,29/07/2025,3653500.0,-2929750.0,200000.0,23000.0
3,1000018721,25V04,28/07/2025,3653500.0,-2929750.0,200000.0,23000.0
4,1000031945,25V04,28/07/2025,3653500.0,-2929750.0,200000.0,23000.0
...,...,...,...,...,...,...,...
3844,98629162,25V04,10/07/2025,3480600.0,-2770300.0,200000.0,23000.0
3845,98653204,25V04,03/07/2025,3480600.0,-2756300.0,200000.0,23000.0
3846,98715576,25V04,19/07/2025,3480600.0,-2756300.0,200000.0,23000.0
3847,98761179,25V04,28/07/2025,3653500.0,-2929750.0,200000.0,23000.0


In [59]:
#aca filtro el dataframe solo con las matriculas que corresponden a los mismo periodos que en etiquetas
ventas_zoho['llave_per_zoho']=ventas_zoho['Documento_Estudiante_zoho'].astype(str)+ventas_zoho['PERIODO'].astype(str)
ventas_zoho = ventas_zoho[ventas_zoho['llave_per_zoho'].isin(Etiquetas_mes_acutual['Llave_etiq_per'])]
ventas_zoho = ventas_zoho.drop('llave_per_zoho', axis=1)
#--------------------------------------------------------------------
ventas_zoho['VALOR_ORDEN']=ventas_zoho['VALOR_ORDEN'].fillna(0)
ventas_zoho['val_becdtos']=ventas_zoho['val_becdtos'].fillna(0)
ventas_zoho['valor_ingles']=ventas_zoho['valor_ingles'].fillna(0)
ventas_zoho['ADICIONALES']=ventas_zoho['ADICIONALES'].fillna(0)
ventas_zoho['VALOR_A_PAGAR']=ventas_zoho['VALOR_ORDEN']+ventas_zoho['val_becdtos']+ventas_zoho['valor_ingles']+ventas_zoho['ADICIONALES']
ventas_zoho_t=ventas_zoho[['Documento_Estudiante_zoho','VALOR_ORDEN','val_becdtos','valor_ingles','ADICIONALES','VALOR_A_PAGAR']]
df_resumen_zoho = ventas_zoho_t.drop_duplicates(subset='Documento_Estudiante_zoho', keep='first')

In [60]:
#ACA AÑADO A Etiquetas_mes_actual_t TODO LO DE ZOHO y lo de Recibos Caja
#----------------------------------------------------------------
Etiquetas_mes_actual_t=Etiquetas_mes_acutual[['Doc_Estudiante','PERIODO','Documento_del_Asesor','Fecha_de_pago','Periodo_Pago','TIPO','EstadoFinal']]
df_resultado1 = pd.merge(Etiquetas_mes_actual_t, df_resumen_zoho, left_on='Doc_Estudiante', right_on='Documento_Estudiante_zoho', how='left')
df_resultado2 = pd.merge(df_resultado1, df_resumen_recibos, left_on='Doc_Estudiante', right_on='CLIENTE', how='left')
df_resultado2.drop(columns=['Documento_Estudiante_zoho','CLIENTE'], inplace=True)
#df_resultado2.to_excel('Etiquetas_ok_MesActual_netoYrecibos.xlsx') #para saber cuanto pago cada estudiante
#------------------DIVIDO ENTRE PREG, POST Y EDU----------------------------------------
df_resultado2_preg=df_resultado2[df_resultado2['TIPO']=='Pregrado'] # ACA SOLO TENGO PREGRADOS
df_resultado2_post=df_resultado2[df_resultado2['TIPO']=='Postgrado'] # ACA SOLO TENGO POSTGRADOS
df_resultado2_Ec=df_resultado2[df_resultado2['TIPO']=='Educacion continuada'] # ACA SOLO TENGO EDUC_CONTINUADA

In [61]:
#-----------CALCULOS EXCLUSIVOS PREGRADO --df_resultado2_preg---------------------------------------
df_resultado2_preg['Pago_Total'] = np.where(df_resultado2_preg['VALOR'] == df_resultado2_preg['VALOR_A_PAGAR'], 'P_Total', 'P_parcial')
df_resultado2_preg['Modalidad'] = df_resultado2_preg['PERIODO'].apply(lambda x: 'VIRTUAL' if 'V' in x else 'PRESENCIAL')
##df_resultado2_preg.to_excel('CuantoPago_Etiq_SoloPreg.xlsx')
almacen_etiquetas_preg_aprov=df_resultado2_preg.copy()
almacen_etiquetas_preg_aprov['Periodo_Pago_1']=PERIODO_ALMACEN
#--------------------------------------------------------------------------------------------------
host_2_12 = '172.16.1.33'
port_2_12 = '1433'
database_2_12 = 'CUN_REPOSITORIO'
url_2_12 = 'mssql+pyodbc://@{host}:{port}/{db}?driver=ODBC Driver 17 for SQL Server'.format(host=host_2_12, port=port_2_12, db=database_2_12)
engine_2_12 = create_engine(url_2_12, fast_executemany=True)

table_name_2_12 = 'PREGR_APROB_CONTAC_MOD2'
schema_name_2_12 = 'COMISIONES_CONTACT'
almacen_etiquetas_preg_aprov.to_sql(
    name=table_name_2_12,
    schema=schema_name_2_12,
    con=engine_2_12,
    if_exists='append',
    index=False,
    chunksize = 10000
)

# End connection
engine_2_12.dispose()

C:\Users\alejandro_arboledas\AppData\Local\Temp\ipykernel_5868\402073692.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_resultado2_preg['Pago_Total'] = np.where(df_resultado2_preg['VALOR'] == df_resultado2_preg['VALOR_A_PAGAR'], 'P_Total', 'P_parcial')
C:\Users\alejandro_arboledas\AppData\Local\Temp\ipykernel_5868\402073692.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_resultado2_preg['Modalidad'] = df_resultado2_preg['PERIODO'].apply(lambda x: 'VIRTUAL' if 'V' in x else 'PRESENCIAL')


In [62]:
df_resultado2_preg['Llave_venta']=df_resultado2_preg['Documento_del_Asesor'].astype(str)+df_resultado2_preg['Pago_Total'].astype(str)+df_resultado2_preg['Modalidad'].astype(str)

#ACA TOMO EL DATAFRAME DE PREGRADOS Y HAGO LOS CONTEOS POR MODALIDAD Y PAGO TOTAL Y LO METO LOS CONTEOS EN UNO NUEVO
#--------------------------------------------------------------------------------------------------------------------
conteo_documentos = df_resultado2_preg['Documento_del_Asesor'].value_counts().reset_index()
conteo_documentos.columns = ['Documento_del_Asesor', 'Frecuencia']
def contar_condiciones(df, documento, pago_total, modalidad):
    return len(df[(df['Documento_del_Asesor'] == documento) & (df['Pago_Total'] == pago_total) & (df['Modalidad'] == modalidad)])
conteo_documentos['P_parcial-PRESENCIAL'] = conteo_documentos['Documento_del_Asesor'].apply(lambda x: contar_condiciones(df_resultado2_preg, x, 'P_parcial', 'PRESENCIAL'))
conteo_documentos['P_Total-PRESENCIAL'] = conteo_documentos['Documento_del_Asesor'].apply(lambda x: contar_condiciones(df_resultado2_preg, x, 'P_Total', 'PRESENCIAL'))
conteo_documentos['P_parcial-VIRTUAL'] = conteo_documentos['Documento_del_Asesor'].apply(lambda x: contar_condiciones(df_resultado2_preg, x, 'P_parcial', 'VIRTUAL'))
conteo_documentos['P_Total-VIRTUAL'] = conteo_documentos['Documento_del_Asesor'].apply(lambda x: contar_condiciones(df_resultado2_preg, x, 'P_Total', 'VIRTUAL'))
#ACA HAGO LOS CALCULOS PARA LLENAR LAS COLUMNAS CON LAS MULTIPLICACIONES
#---------------------------------------------------------------------------------------
conteo_documentos['Vent_Parc_PRES'] = 0
conteo_documentos.loc[(conteo_documentos['Frecuencia'] >= 30) & (conteo_documentos['Frecuencia'] < 40), 'Vent_Parc_PRES'] = conteo_documentos['P_parcial-PRESENCIAL'] * 49980
conteo_documentos.loc[(conteo_documentos['Frecuencia'] >= 40) & (conteo_documentos['Frecuencia'] < 50), 'Vent_Parc_PRES'] = conteo_documentos['P_parcial-PRESENCIAL'] * 66640
conteo_documentos.loc[(conteo_documentos['Frecuencia'] >= 50) & (conteo_documentos['Frecuencia'] < 60), 'Vent_Parc_PRES'] = conteo_documentos['P_parcial-PRESENCIAL'] * 99960
conteo_documentos.loc[(conteo_documentos['Frecuencia'] >= 60) & (conteo_documentos['Frecuencia'] < 70), 'Vent_Parc_PRES'] = conteo_documentos['P_parcial-PRESENCIAL'] * 116620
conteo_documentos.loc[conteo_documentos['Frecuencia'] >= 70, 'Vent_Parc_PRES'] = conteo_documentos['P_parcial-PRESENCIAL'] * 133280
conteo_documentos['Vent_Total_PRES'] = 0
conteo_documentos.loc[(conteo_documentos['Frecuencia'] >= 30) & (conteo_documentos['Frecuencia'] < 40), 'Vent_Total_PRES'] = conteo_documentos['P_Total-PRESENCIAL'] * 88000
conteo_documentos.loc[(conteo_documentos['Frecuencia'] >= 40) & (conteo_documentos['Frecuencia'] < 50), 'Vent_Total_PRES'] = conteo_documentos['P_Total-PRESENCIAL'] * 110000
conteo_documentos.loc[(conteo_documentos['Frecuencia'] >= 50) & (conteo_documentos['Frecuencia'] < 60), 'Vent_Total_PRES'] = conteo_documentos['P_Total-PRESENCIAL'] * 132000
conteo_documentos.loc[(conteo_documentos['Frecuencia'] >= 60) & (conteo_documentos['Frecuencia'] < 70), 'Vent_Total_PRES'] = conteo_documentos['P_Total-PRESENCIAL'] * 154000
conteo_documentos.loc[conteo_documentos['Frecuencia'] >= 70, 'Vent_Total_PRES'] = conteo_documentos['P_Total-PRESENCIAL'] * 176000
conteo_documentos['Vent_Parc_VIRT'] = 0
conteo_documentos.loc[(conteo_documentos['Frecuencia'] >= 30) & (conteo_documentos['Frecuencia'] < 40), 'Vent_Parc_VIRT'] = conteo_documentos['P_parcial-VIRTUAL'] * 45000
conteo_documentos.loc[(conteo_documentos['Frecuencia'] >= 40) & (conteo_documentos['Frecuencia'] < 50), 'Vent_Parc_VIRT'] = conteo_documentos['P_parcial-VIRTUAL'] * 55000
conteo_documentos.loc[(conteo_documentos['Frecuencia'] >= 50) & (conteo_documentos['Frecuencia'] < 60), 'Vent_Parc_VIRT'] = conteo_documentos['P_parcial-VIRTUAL'] * 65000
conteo_documentos.loc[(conteo_documentos['Frecuencia'] >= 60) & (conteo_documentos['Frecuencia'] < 70), 'Vent_Parc_VIRT'] = conteo_documentos['P_parcial-VIRTUAL'] * 75000
conteo_documentos.loc[conteo_documentos['Frecuencia'] >= 70, 'Vent_Parc_VIRT'] = conteo_documentos['P_parcial-VIRTUAL'] * 85000
conteo_documentos['Vent_Total_VIRT'] = 0
conteo_documentos.loc[(conteo_documentos['Frecuencia'] >= 30) & (conteo_documentos['Frecuencia'] < 40), 'Vent_Total_VIRT'] = conteo_documentos['P_Total-VIRTUAL'] * 55200
conteo_documentos.loc[(conteo_documentos['Frecuencia'] >= 40) & (conteo_documentos['Frecuencia'] < 50), 'Vent_Total_VIRT'] = conteo_documentos['P_Total-VIRTUAL'] * 69000
conteo_documentos.loc[(conteo_documentos['Frecuencia'] >= 50) & (conteo_documentos['Frecuencia'] < 60), 'Vent_Total_VIRT'] = conteo_documentos['P_Total-VIRTUAL'] * 82800
conteo_documentos.loc[(conteo_documentos['Frecuencia'] >= 60) & (conteo_documentos['Frecuencia'] < 70), 'Vent_Total_VIRT'] = conteo_documentos['P_Total-VIRTUAL'] * 96600
conteo_documentos.loc[conteo_documentos['Frecuencia'] >= 70, 'Vent_Total_VIRT'] = conteo_documentos['P_Total-VIRTUAL'] * 110400
conteo_documentos['Total_Com_Preg']= conteo_documentos['Vent_Parc_PRES'] + conteo_documentos['Vent_Total_PRES'] + conteo_documentos['Vent_Parc_VIRT'] + conteo_documentos['Vent_Total_VIRT']
#conteo_documentos.to_excel('Tabla_conteos_preg_total.xlsx')

C:\Users\alejandro_arboledas\AppData\Local\Temp\ipykernel_5868\811235510.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_resultado2_preg['Llave_venta']=df_resultado2_preg['Documento_del_Asesor'].astype(str)+df_resultado2_preg['Pago_Total'].astype(str)+df_resultado2_preg['Modalidad'].astype(str)


In [63]:
#-----------CALCULOS EXCLUSIVOS POST--df_resultado2_post---------------------------------------
df_resultado2_post['Pago_Total'] = np.where(df_resultado2_post['VALOR'] == df_resultado2_post['VALOR_A_PAGAR'], 'P_Total', 'P_parcial')
##df_resultado2_preg.to_excel('CuantoPago_Etiq_SoloPreg.xlsx')
almacen_etiquetas_post_aprov=df_resultado2_post.copy()
almacen_etiquetas_post_aprov['Periodo_Pago_1']=PERIODO_ALMACEN

C:\Users\alejandro_arboledas\AppData\Local\Temp\ipykernel_5868\2123383.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_resultado2_post['Pago_Total'] = np.where(df_resultado2_post['VALOR'] == df_resultado2_post['VALOR_A_PAGAR'], 'P_Total', 'P_parcial')


In [64]:
#SUBO LA DATA DE POSGRADOS APROBADOS

host_2_12 = '172.16.1.33'
port_2_12 = '1433'
database_2_12 = 'CUN_REPOSITORIO'
url_2_12 = 'mssql+pyodbc://@{host}:{port}/{db}?driver=ODBC Driver 17 for SQL Server'.format(host=host_2_12, port=port_2_12, db=database_2_12)
engine_2_12 = create_engine(url_2_12, fast_executemany=True)

table_name_2_12 = 'POSG_APROB_CONTAC_MOD2'
schema_name_2_12 = 'COMISIONES_CONTACT'
almacen_etiquetas_post_aprov.to_sql(
    name=table_name_2_12,
    schema=schema_name_2_12,
    con=engine_2_12,
    if_exists='append',
    index=False,
    chunksize = 10000
)

# End connection
engine_2_12.dispose()

In [65]:


#ACA TOMO EL DATAFRAME DE POSGRADOS Y HAGO LOS CONTEOS POR PAGO TOTAL Y LO METO LOS CONTEOS EN UNO NUEVO
#--------------------------------------------------------------------------------------------------------------------
#-----------------------------------------------------
conteo_documentos_post = df_resultado2_post['Documento_del_Asesor'].value_counts().reset_index()
conteo_documentos_post.columns = ['Documento_del_Asesor_post', 'Ventas_post']
def contar_condiciones_pos(df, documento, pago_total):
    return len(df[(df['Documento_del_Asesor'] == documento) & (df['Pago_Total'] == pago_total)])
conteo_documentos_post['P_parcial'] = conteo_documentos_post['Documento_del_Asesor_post'].apply(lambda x: contar_condiciones_pos(df_resultado2_post, x, 'P_parcial'))
conteo_documentos_post['P_Total'] = conteo_documentos_post['Documento_del_Asesor_post'].apply(lambda x: contar_condiciones_pos(df_resultado2_post, x, 'P_Total'))

#ACA HAGO LOS CALCULOS PARA LLENAR LAS COLUMNAS CON LAS MULTIPLICACIONES
#---------------------------------------------------------------------------------------
conteo_documentos_post['Vent_Parc'] = 0
conteo_documentos_post.loc[(conteo_documentos_post['Ventas_post'] >= 30) & (conteo_documentos_post['Ventas_post'] < 40), 'Vent_Parc'] = conteo_documentos_post['P_parcial'] * 50400
conteo_documentos_post.loc[(conteo_documentos_post['Ventas_post'] >= 40) & (conteo_documentos_post['Ventas_post'] < 50), 'Vent_Parc'] = conteo_documentos_post['P_parcial'] * 64800
conteo_documentos_post.loc[(conteo_documentos_post['Ventas_post'] >= 50) & (conteo_documentos_post['Ventas_post'] < 60), 'Vent_Parc'] = conteo_documentos_post['P_parcial'] * 79200
conteo_documentos_post.loc[(conteo_documentos_post['Ventas_post'] >= 60) & (conteo_documentos_post['Ventas_post'] < 70), 'Vent_Parc'] = conteo_documentos_post['P_parcial'] * 93600
conteo_documentos_post.loc[conteo_documentos_post['Ventas_post'] >= 70, 'Vent_Parc'] = conteo_documentos_post['P_parcial'] * 108000
conteo_documentos_post['Vent_Total'] = 0
conteo_documentos_post.loc[(conteo_documentos_post['Ventas_post'] >= 30) & (conteo_documentos_post['Ventas_post'] < 40), 'Vent_Total'] = conteo_documentos_post['P_Total'] * 112000
conteo_documentos_post.loc[(conteo_documentos_post['Ventas_post'] >= 40) & (conteo_documentos_post['Ventas_post'] < 50), 'Vent_Total'] = conteo_documentos_post['P_Total'] * 144000
conteo_documentos_post.loc[(conteo_documentos_post['Ventas_post'] >= 50) & (conteo_documentos_post['Ventas_post'] < 60), 'Vent_Total'] = conteo_documentos_post['P_Total'] * 176000
conteo_documentos_post.loc[(conteo_documentos_post['Ventas_post'] >= 60) & (conteo_documentos_post['Ventas_post'] < 70), 'Vent_Total'] = conteo_documentos_post['P_Total'] * 208000
conteo_documentos_post.loc[conteo_documentos_post['Ventas_post'] >= 70, 'Vent_Total'] = conteo_documentos_post['P_Total'] * 240000
conteo_documentos_post['Total_Com_Pos']= conteo_documentos_post['Vent_Parc'] + conteo_documentos_post['Vent_Total'] 

In [66]:
#CALCULO DE POSTGRADOS ANTIGUO (VIEJO MARZO)
#-----------------------------------------------------
#conteo_documentos_post = df_resultado2_post['Documento_del_Asesor'].value_counts().reset_index()
#conteo_documentos_post.columns = ['Documento_del_Asesor_post', 'Ventas_post']
#conteo_documentos_post.to_excel('VentasPost.xlsx')
#def calcular_valor_ventas(cantidad):
#    if cantidad == 0:
#        return 0
#    elif 1 <= cantidad < 6:
#        return cantidad * 20000
#    elif 6 <= cantidad < 11:
#        return cantidad * 40000
#    elif 11 <= cantidad < 21:
#        return cantidad * 60000
#    elif cantidad >= 21:
#        return cantidad * 90000
#conteo_documentos_post['Valor_Vent_post'] = conteo_documentos_post['Ventas_post'].apply(calcular_valor_ventas)
#conteo_documentos_post.to_excel('VentasPostN.xlsx')
#-----------------------------------------------------
#CALCULO EDUCACION CONTINUADA
#-----------------------------------------------------
conteo_documentos_Ec = df_resultado2_Ec['Documento_del_Asesor'].value_counts().reset_index()
conteo_documentos_Ec.columns = ['Documento_del_Asesor_Ec', 'Ventas_Ec']
#conteo_documentos_Ec.to_excel('Ventas_Ec.xlsx')
def calcular_valor_ventas_Ec(cantidad):
    if cantidad == 0:
        return 0
    elif cantidad >= 1:
        return cantidad * 10000
conteo_documentos_Ec['Valor_Vent_Ec'] = conteo_documentos_Ec['Ventas_Ec'].apply(calcular_valor_ventas_Ec)
#conteo_documentos_Ec.to_excel('Ventas_EcN.xlsx')

In [67]:
#ACA AÑADO AL DATAFRAME DE PREGRADOS POSTGRADOS
#------------------------------------------------------------------
resultado_m = pd.merge(conteo_documentos, conteo_documentos_post, left_on='Documento_del_Asesor', right_on='Documento_del_Asesor_post', how='outer')
resultado_m['Documento_Combinado'] = resultado_m['Documento_del_Asesor'].combine_first(resultado_m['Documento_del_Asesor_post'])
resultado_m.drop(columns=['Documento_del_Asesor', 'Documento_del_Asesor_post'], inplace=True)
resultado_m=resultado_m.fillna(0)
columnas = resultado_m.columns.tolist()
columnas = ['Documento_Combinado'] + [col for col in columnas if col != 'Documento_Combinado']
resultado_m = resultado_m[columnas]
resultado_m.rename(columns={'Documento_Combinado': 'Documento_del_Asesor'}, inplace=True)
#resultado_m.to_excel('Merge_totalC_C.xlsx')
#------------------------------------------------------------------
#ACA AÑADO AL DATAFRAME DE UNIDO EDUCACION CONTINUADA
#------------------------------------------------------------------
resultado_f = pd.merge(resultado_m, conteo_documentos_Ec, left_on='Documento_del_Asesor', right_on='Documento_del_Asesor_Ec', how='outer')
resultado_f['Documento_Combinado'] = resultado_f['Documento_del_Asesor'].combine_first(resultado_f['Documento_del_Asesor_Ec'])
resultado_f.drop(columns=['Documento_del_Asesor', 'Documento_del_Asesor_Ec'], inplace=True)
resultado_f=resultado_f.fillna(0)
columnas1 = resultado_f.columns.tolist()
columnas1 = ['Documento_Combinado'] + [col1 for col1 in columnas1 if col1 != 'Documento_Combinado']
resultado_f = resultado_f[columnas1]
#------------------------------------------------------------------
#ENCUENTRO EL TOTAL TOTAL
#------------------------------------------------------------------
resultado_f['TOTAL_COMISION_GENERAL']=resultado_f['Total_Com_Preg']+resultado_f['Total_Com_Pos']+resultado_f['Valor_Vent_Ec']
##resultado_f.to_excel('Merge_totalC_f.xlsx') #Ya tengo aqui mi tabla de ventas totales


In [68]:
resultado_f.rename(columns={'Total_Com_Pos':'Valor_Vent_post','P_parcial':'P_parcial_posg','P_Total':'P_Total_posg',
                            'Vent_Parc':'Vent_Parc_posg', 'Vent_Total' : 'Vent_Total_posg'}, inplace=True)

In [69]:
#AQUI INYECTO LA BASE DESGLOSADA COMPLETA POR QUE CUANDO EJECUTO EL PYTHON DE RELIQUIDACIONES SE NECESITA
resultado_almacen=resultado_f.copy()
resultado_almacen['Periodo_Pago']=PERIODO_ALMACEN
#-------------------------------------------------------------------
host_2_12 = '172.16.1.33'
port_2_12 = '1433'
database_2_12 = 'CUN_REPOSITORIO'
url_2_12 = 'mssql+pyodbc://@{host}:{port}/{db}?driver=ODBC Driver 17 for SQL Server'.format(host=host_2_12, port=port_2_12, db=database_2_12)
engine_2_12 = create_engine(url_2_12, fast_executemany=True)

table_name_2_12 = 'FIJA_LIQ_ACTUAL_DESGLOS'
schema_name_2_12 = 'COMISIONES_CONTACT'
resultado_almacen.to_sql(
    name=table_name_2_12,
    schema=schema_name_2_12,
    con=engine_2_12,
    if_exists='append',
    index=False,
    chunksize = 10000
)

# End connection
engine_2_12.dispose()

In [70]:
#AQUI YA FORMO LA TABLA QUE NECESITA EL SP
#------------------------------------------------
Tabla_Conteos_SP=resultado_f.copy()
Tabla_Conteos_SP['Periodo_Pago']=PERIODO_NORMAL_float
Tabla_Conteos_SP=Tabla_Conteos_SP[['Documento_Combinado','Periodo_Pago','Frecuencia','Ventas_post','Ventas_Ec','Total_Com_Preg','Valor_Vent_post','Valor_Vent_Ec','TOTAL_COMISION_GENERAL']] 
Tabla_Conteos_SP.rename(columns={'Documento_Combinado': 'Documento_del_Asesor',
                            'Frecuencia': 'Pregrados',
                            'Ventas_post': 'Postgrado',
                            'Ventas_Ec': 'Educacion_continuada',
                            'Total_Com_Preg': 'Valor_Pregrado',
                            'Valor_Vent_post': 'Valor_Postgrado',
                            'Valor_Vent_Ec': 'Valor_Educacion_continuada',
                            'TOTAL_COMISION_GENERAL': 'Total_Comision',
                            }, inplace=True)
#Tabla_Conteos_SP.to_excel('Tabla conteos pal sp.xlsx')#Esta es la tabla que se inyecta a SQL

In [71]:
Tabla_Conteos_SP.dtypes
#AQUI SIMPLEMENTE VALIDE QUE TIPOS DE DATOSE IBA A METER A LA TABLA
# df_dtypesTabla_Conteos_SP = pd.DataFrame(Tabla_Conteos_SP.dtypes).reset_index()
# df_dtypesTabla_Conteos_SP.columns = ['Columna', 'Tipo de Dato']
# df_dtypesTabla_Conteos_SP.to_excel('TIPOS CONTEOS.xlsx')

Documento_del_Asesor           object
Periodo_Pago                   object
Pregrados                     float64
Postgrado                     float64
Educacion_continuada          float64
Valor_Pregrado                float64
Valor_Postgrado               float64
Valor_Educacion_continuada    float64
Total_Comision                float64
dtype: object

In [72]:
host_2_12 = '172.16.1.33'
port_2_12 = '1433'
database_2_12 = 'CUN_REPOSITORIO'
url_2_12 = 'mssql+pyodbc://@{host}:{port}/{db}?driver=ODBC Driver 17 for SQL Server'.format(host=host_2_12, port=port_2_12, db=database_2_12)
engine_2_12 = create_engine(url_2_12, fast_executemany=True)

table_name_2_12 = 'TEMP_CONTEO_VENTAS'
schema_name_2_12 = 'COMISIONES_CONTACT'
# Send the data to sql server
Tabla_Conteos_SP.to_sql(
    name=table_name_2_12,
    schema=schema_name_2_12,
    con=engine_2_12,
    if_exists='replace',
    index=False,
    chunksize = 10000
)

# End connection
engine_2_12.dispose()

In [73]:
#CONSULTA PARA CREAR LA TABLA TEMPORAL EN SQL 
with engine.connect() as connection:
    try:

        sql_query = text(f'''
        IF object_id('CUN_REPOSITORIO.COMISIONES_CONTACT.TEMP_AFECTACIONES') IS NOT NULL
        DROP TABLE CUN_REPOSITORIO.COMISIONES_CONTACT.TEMP_AFECTACIONES

        Select *, Calidad_Resultado+Formacion_Resultado+Ventas_Resultado+Adherencia_Resultado+Conversion_Resultado+Productividad_Resultado AS Por_Afectacion
        INTO CUN_REPOSITORIO.COMISIONES_CONTACT.TEMP_AFECTACIONES 
        from(
            Select Documento_del_Asesor,Correo_Asesor,Periodo,
            case when CALIDAD <> 'Cumple' then 0.07 else 0 end as Calidad_Resultado,
            case when [FORMACION ]  <> 'Cumple' then 0.07 else 0 end as Formacion_Resultado,
            case when VENTAS  <> 'Cumple' then 0.07 else 0 end as Ventas_Resultado,
            case when ADHERENCIA  <> 'Cumple' then 0.07 else 0 end as Adherencia_Resultado,
            case when CONVERSION  <> 'Cumple' then 0.07 else 0 end as Conversion_Resultado,
            case when PRODUCTIVIDAD  <> 'Cumple' then 0.07 else 0 end as Productividad_Resultado
        from CUN_REPOSITORIO.Comisiones_Contact.Afectaciones_Contact)t1
        ''')
        connection.execute(sql_query)
        connection.commit()  # Asegurar que los cambios se guarden en la base de datos
        print("Consulta realizada con exito")
    except Exception as e:
        print(f"Error al eliminar registros: {e}")
    finally:
        connection.close()

Consulta realizada con exito


In [74]:
import datetime

# Obtener la fecha actual
hoy = datetime.date.today()

# Calcular el primer día del mes actual y restar un día para obtener el último día del mes anterior
primer_dia_mes_actual = hoy.replace(day=1)
ultimo_dia_mes_anterior = primer_dia_mes_actual - datetime.timedelta(days=1)

# Extraer mes y año del mes anterior
mes_anterior = ultimo_dia_mes_anterior.month-1
anio_anterior = ultimo_dia_mes_anterior.year

# Asignar variables
MES_ACTUAL_EN_RELIQUIDACION = f"'{anio_anterior}{mes_anterior:02d}'"

# Imprimir resultados
print("Mes reliquidar =", MES_ACTUAL_EN_RELIQUIDACION)


Mes reliquidar = '202506'


In [75]:
#Importacion de librerias
import requests
import pandas as pd
from unidecode import unidecode
import numpy as np
from sqlalchemy import create_engine, text, inspect
from sqlalchemy.orm import sessionmaker

In [76]:
# Configura los detalles de la conexión
host='172.16.1.33'
port='1433'
database= 'CUN_REPOSITORIO'

try:
    # Crea la cadena de conexión
   url = 'mssql+pyodbc://@{host}:{port}/{db}?driver=SQL+Server'.format(host=host, port=port, db=database)

   
    # Crear conexion con base de datos
   engine = create_engine(url) 
   print("Conexion a la base de datos realizada") 
    
except Exception as e:
    print("Error:", e)

Conexion a la base de datos realizada


In [77]:
#CONSULTA PARA CREAR LA TABLA TEMPORAL EN SQL 
with engine.connect() as connection:
    try:

        sql_query = text(f'''
         IF object_id('CUN_REPOSITORIO.COMISIONES_CONTACT.TEMP_BASE_RELIQUIDACION') IS NOT NULL
            DROP TABLE CUN_REPOSITORIO.COMISIONES_CONTACT.TEMP_BASE_RELIQUIDACION
        Select t1.Doc_Estudiante,t1.PERIODO,t1.Documento_del_Asesor,t1.Propietario_del_registro,t1.Programa_vendido,t1.Correo_Asesor,t1.Periodo_Pago
            ,t3.EST_MATRICULADO,t3.NCR_ANULACION,t3.TIPO,t3.ORDEN,
            t3.VAL_LIQUIDADO,t3.NUEVO,t3.ESTADO_PAGO,t3.Doc_Alum as Documento_Estudiante_zoho
        INTO CUN_REPOSITORIO.COMISIONES_CONTACT.TEMP_BASE_RELIQUIDACION
        from Comisiones_Contact.Etiquetas t1
                Left join (Select * from (
        select Doc_Alum,ESTADO_PAGO,NCR_ANULACION,NUEVO,EST_MATRICULADO,
                PERIODO,ORDEN,VAL_LIQUIDADO,TIPO
                ,row_number() over (partition by Doc_Alum,PERIODO  order by ESTADO_PAGO desc,NCR_ANULACION desc, EST_MATRICULADO,ORDEN asc ) cont
                            from ##Ventas where DOCUMENTO in ( 'FAMA','FECO')  and PERIODO not like '%I%'
                                                                            /* and EST_MATRICULADO = '1-Activo'
                                                                                and NUEVO = 'NUEVO'
                                                                                AND NCR_ANULACION = 0
            AND ESTADO_PAGO = 'PAGO'*/)t1 where cont = 1) t3 on t1.Doc_Estudiante = t3.Doc_Alum and t1.PERIODO = t3.PERIODO
                where Periodo_Pago = {MES_ACTUAL_EN_RELIQUIDACION} and EstadoFinal <> 'ok'
        ''')
        connection.execute(sql_query)
        connection.commit()  # Asegurar que los cambios se guarden en la base de datos
        print("Consulta realizada con exito")
    except Exception as e:
        print(f"Error al eliminar registros: {e}")
    finally:
        connection.close()

Consulta realizada con exito


In [78]:
#CONSULTA PARA CREAR LA TABLA TEMPORAL EN SQL 
with engine.connect() as connection:
    try:

        sql_query = text(f'''
                 Delete from Comisiones_Contact.EtiquetasReliquidacion where Periodo_Pago = {MES_ACTUAL_EN_RELIQUIDACION} 

                INSERT INTO Comisiones_Contact.EtiquetasReliquidacion
                select *,
                    case
                            when Motivo =  'OK' THEN 'OK'
                ELSE Motivo END AS EstadoFinal
                    from (select distinct t1.*,
                                            case
                                                when t1.Documento_Estudiante_zoho is null
                                                    then 'No coincide con Iceberg o Sinu'
                                                when t1.ESTADO_PAGO <> 'PAGO' then 'No Registra Pago'
                                                when t1.ESTADO_PAGO IS NULL then 'No Registra Pago'
                                                when t1.EST_MATRICULADO IS NULL then 'No Registra Matricula'
                                                --  when t1.NCR_ANULACION < 0 then 'Tiene Anulación'
                                                when t1.NUEVO = 'ANTIGUO' and TIPO <> 'Educacion continuada' then 'Estudiante ANTIGUO'
                                                when t1.EST_MATRICULADO <> '1-Activo'
                                                    then 'Estado Diferente al Activo ' + t1.EST_MATRICULADO
                                                when t1.VAL_LIQUIDADO < 200000 then 'Valor liquidado Menor 20%'
                                                else 'ok' end Motivo,
                                            t2.Propietario_de_Fase_Name--,t2.Correo_electronico_asesor
                            from CUN_REPOSITORIO.COMISIONES_CONTACT.TEMP_BASE_RELIQUIDACION t1
                                        left join CUN_REPOSITORIO.COMISIONES_CONTACT.TEMP_ZOHO_FUNELL t2
                                                on t1.Doc_Estudiante = t2.Número_de_Documento and t1.PERIODO = t2.Periodo and
                                                    t1.Correo_Asesor = t2.Propietario_de_Fase_Name
                            )t1
        ''')
        connection.execute(sql_query)
        connection.commit()  # Asegurar que los cambios se guarden en la base de datos
        print("Consulta realizada con exito")
    except Exception as e:
        print(f"Error al eliminar registros: {e}")
    finally:
        connection.close()

Consulta realizada con exito


In [79]:
host2_1 = '172.16.1.33'
port2_1 = '1433'
database2_1 = 'CUN_REPOSITORIO'

try:
    url2_1 = 'mssql+pyodbc://@{host}:{port}/{db}?trusted_connection=yes&driver=SQL+Server'.format(host=host2_1, port=port2_1, db=database2_1)
    engine2_1 = create_engine(url2_1) 
    print("Conexion a la base de datos realizada")
    #archivo_orig = pd.read_sql("select * from CUN_REPOSITORIO.Comisiones.Com_Telec_Oct_2024_2_yCT", engine2_1)
    Etiquetas_penultimo_paso = pd.read_sql(f'''
        select * from CUN_REPOSITORIO.COMISIONES_CONTACT.EtiquetasReliquidacion
                                           WHERE Periodo_Pago = {MES_ACTUAL_EN_RELIQUIDACION}  and EstadoFinal ='OK'
                               ''', engine2_1)
    Consulta_z2 = Etiquetas_penultimo_paso.copy()
        
except Exception as e:
    print("Error:", e)
finally:
    # Cerrar la conexión
    if 'engine2_1' in locals():
        engine2_1.dispose()
        print("Conexión cerrada")
#-------------------------------------------------------------------------------------------
Etiquetas_penultimo_paso['Doc_Estudiante'] = Etiquetas_penultimo_paso['Doc_Estudiante'].astype(int).astype(str)
Etiquetas_penultimo_paso['PERIODO'] = Etiquetas_penultimo_paso['PERIODO'].astype(str)
Etiquetas_penultimo_paso['Llave_per'] = Etiquetas_penultimo_paso['Doc_Estudiante'] + Etiquetas_penultimo_paso['PERIODO']
#--------------------------------------------------------------------------------------------
Etiquetas_penultimo_paso.shape
#--------------------------------------------------------------------------------------------
#--------------------------------------------------------------------------------------------

Conexion a la base de datos realizada
Conexión cerrada


(54, 19)

In [80]:
#Eliminamos los registros de la tabla para luego insertarlos nuevamente
with engine.connect() as connection:
    try:

        sql_query = text(f'''
                DELETE FROM CUN_REPOSITORIO.COMISIONES_CONTACT.EtiquetasReliquidacion WHERE Periodo_Pago = {MES_ACTUAL_EN_RELIQUIDACION} and EstadoFinal ='OK'
        ''')
        connection.execute(sql_query)
        connection.commit()  # Asegurar que los cambios se guarden en la base de datos
        print("Consulta realizada con exito")
    except Exception as e:
        print(f"Error al eliminar registros: {e}")
    finally:
        connection.close()

Consulta realizada con exito


In [81]:
para_orden = Etiquetas_penultimo_paso['Doc_Estudiante'].unique()
valores_unicos_df1_orden = "', '".join(map(str, para_orden))
valores_unicos_df1_orden = f"'{valores_unicos_df1_orden}'"
#----------------------------------------------------------------------------------
host2_1 = '172.16.1.33'
port2_1 = '1433'
database2_1 = 'CUN_REPOSITORIO'

try:
    url2_1 = 'mssql+pyodbc://@{host}:{port}/{db}?trusted_connection=yes&driver=SQL+Server'.format(host=host2_1, port=port2_1, db=database2_1)
    engine2_1 = create_engine(url2_1) 
    print("Conexion a la base de datos realizada")
    df_ordenes= pd.read_sql(f'''
SELECT DISTINCT Documento_Estudiante_zoho, 
    PERIODO, 
    COD_UNI,
    FuerzaComercialFinal,
    NUEVO,
    CLASE_ACTUAL
FROM CUN_REPOSITORIO.DBO.RECIBOS_CONTACT2_SIN_ZOHO
     WHERE Documento_Estudiante_zoho IN ({valores_unicos_df1_orden})
       AND ESTADO_PAGO = 'PAGO'
       AND EST_MATRICULADO = '1-Activo'
                            ''', engine2_1)
    df_ordenes2 = df_ordenes.copy()
        
except Exception as e:
    print("Error:", e)
finally:
    # Cerrar la conexión
    if 'engine2_1' in locals():
        engine2_1.dispose()
        print("Conexión cerrada")
df_ordenes['llave_orden_sql']=df_ordenes['Documento_Estudiante_zoho'].astype(str) + df_ordenes['PERIODO'].astype(str)
df_ordenes['llave_total_cod_uni']=df_ordenes['Documento_Estudiante_zoho'].astype(str) + df_ordenes['COD_UNI'].astype(str)

Conexion a la base de datos realizada
Conexión cerrada


In [82]:
#ARMO UNA MINI CONSULTA PERO SOLO PARA VALIDAR COD_UNI EN EL PERIODO ACTUAL
df_recibos_per_Actual=df_ordenes[df_ordenes['llave_orden_sql'].isin(Etiquetas_penultimo_paso['Llave_per'])]
df_recibos_per_Actual=df_recibos_per_Actual.drop_duplicates(subset='llave_orden_sql', keep='first')
#-----------------EXPORTANDO EVIDENCIA--------------------------------------
from datetime import datetime
fecha_hoy_rc = datetime.now().strftime('%Y-%m-%d')
df_recibos_per_Actual.to_excel(f'Evidencia_Recibos_cont_per_actual{fecha_hoy_rc}_Reliq.xlsx')
#-------------------------------


In [83]:
Etiquetas_penultimo_paso = Etiquetas_penultimo_paso.merge(df_recibos_per_Actual[['llave_orden_sql', 'COD_UNI']], 
                                                left_on='Llave_per', 
                                                right_on='llave_orden_sql', 
                                                how='left')
Etiquetas_penultimo_paso.drop(columns=['llave_orden_sql'], inplace=True)
Etiquetas_penultimo_paso.shape

(54, 20)

In [84]:
para_orden = Etiquetas_penultimo_paso['Doc_Estudiante'].unique()
valores_unicos_df1_orden = "', '".join(map(str, para_orden))
valores_unicos_df1_orden = f"'{valores_unicos_df1_orden}'"
#----------------------------------------------------------------------------------
host2_1 = '172.16.1.33'
port2_1 = '1433'
database2_1 = 'CUN_REPOSITORIO'

try:
    url2_1 = 'mssql+pyodbc://@{host}:{port}/{db}?trusted_connection=yes&driver=SQL+Server'.format(host=host2_1, port=port2_1, db=database2_1)
    engine2_1 = create_engine(url2_1) 
    print("Conexion a la base de datos realizada")
    df_histo_telec= pd.read_sql(f'''
SELECT DISTINCT DOC_ALUM, PERIODO, COD_UNI FROM CUN_REPOSITORIO.FINANCIERA.HISTORICO_COM_TELEC_2025
     WHERE DOC_ALUM IN ({valores_unicos_df1_orden})
                            ''', engine2_1)
        
except Exception as e:
    print("Error:", e)
finally:
    # Cerrar la conexión
    if 'engine2_1' in locals():
        engine2_1.dispose()
        print("Conexión cerrada")
df_histo_telec['llave_hist_tel_cod_uni']=df_histo_telec['DOC_ALUM'].astype(str) + df_histo_telec['COD_UNI'].astype(str)
df_histo_telec['llave_hist_CC_PER']=df_histo_telec['DOC_ALUM'].astype(str) + df_histo_telec['PERIODO'].astype(str)

df_histo_telec
#--------------------------------
#--------------------------------

Conexion a la base de datos realizada
Conexión cerrada


,DOC_ALUM,PERIODO,COD_UNI,llave_hist_tel_cod_uni,llave_hist_CC_PER
0,1000616338,25E1T,ESP19,1000616338ESP19,100061633825E1T
1,1001098752,25V04,VTE01,1001098752VTE01,100109875225V04
2,1022994511,25ES2,ESP23,1022994511ESP23,102299451125ES2
3,1030606362,25V04,PVC10,1030606362PVC10,103060636225V04
4,1056954677,25V04,VTE01,1056954677VTE01,105695467725V04
5,1070943713,25ES2,ESP20,1070943713ESP20,107094371325ES2
6,1078246218,25ES2,ESP24,1078246218ESP24,107824621825ES2
7,22742579,25V04,VTE01,22742579VTE01,2274257925V04


In [85]:
Etiquetas_penultimo_paso['Esta_en_hist_telec_x_per']=Etiquetas_penultimo_paso['Llave_per'].isin(df_histo_telec['llave_hist_CC_PER']).astype(int)
Etiquetas_penultimo_paso.loc[Etiquetas_penultimo_paso['Esta_en_hist_telec_x_per']==1, 'EstadoFinal'] = 'ENTRO CON OTRA FUERZA_T'
#----------------------------------

In [86]:
#ARMO LLAVE POR COD UNI PARA TODOS
Etiquetas_penultimo_paso['Llave_liq_cod_uni']=Etiquetas_penultimo_paso['Doc_Estudiante'] + Etiquetas_penultimo_paso['COD_UNI']

"Divido mi liquidacion entre los que si obtuvieron su cod_uni y los que no"
Etiquetas_penultimo_paso_sin_cod_uni=Etiquetas_penultimo_paso[Etiquetas_penultimo_paso['COD_UNI'].isna()]
Etiquetas_penultimo_Con_cod_uni=Etiquetas_penultimo_paso[Etiquetas_penultimo_paso['COD_UNI'].notna()]

In [87]:
condicion_histor_telec_novedades=(
    (df_histo_telec['COD_UNI'].isna())|
    (df_histo_telec['COD_UNI']=='0')|
    (df_histo_telec['COD_UNI']==0)|
    (df_histo_telec['COD_UNI']=='Null')|
    (df_histo_telec['COD_UNI']=='null')|
    (df_histo_telec['COD_UNI']=='NULL')|
    (df_histo_telec['COD_UNI']=='')|
    (df_histo_telec['COD_UNI']==' ')|
    (df_histo_telec['COD_UNI']=='na')    
)

#divido mi historico telecampus en dos, que tiene COD_UNI Valido y los que no
df_histo_telec_SOLO_NOvedades=df_histo_telec[condicion_histor_telec_novedades]
df_histo_telec_Depurado=df_histo_telec[~condicion_histor_telec_novedades]

#ETIQUETEO MI LIQUIDACION CON COD_UNI
# VALIDO CONTRA EL HISTORICO telec de novedades por cc y contra el historico tele depurado por llave
Etiquetas_penultimo_Con_cod_uni['Esta_En_histor_telec_depur']=Etiquetas_penultimo_Con_cod_uni['Llave_liq_cod_uni'].isin(df_histo_telec_Depurado['llave_hist_tel_cod_uni']).astype(int)
Etiquetas_penultimo_Con_cod_uni['Esta_En_histor_telec_NOved']=Etiquetas_penultimo_Con_cod_uni['Doc_Estudiante'].isin(df_histo_telec_SOLO_NOvedades['DOC_ALUM']).astype(int)

condicion_cod_uni_liq=((Etiquetas_penultimo_Con_cod_uni['Esta_En_histor_telec_depur']==1)|(Etiquetas_penultimo_Con_cod_uni['Esta_En_histor_telec_NOved']==1))
Etiquetas_penultimo_Con_cod_uni.loc[condicion_cod_uni_liq, 'EstadoFinal'] = 'ENTRO CON OTRA FUERZA_T'

#ETIQUETO LOS QUE NO TIENEN COD UNI
Etiquetas_penultimo_paso_sin_cod_uni['Esta_En_histor_telec_TOTAL']=Etiquetas_penultimo_paso_sin_cod_uni['Doc_Estudiante'].isin(df_histo_telec['DOC_ALUM']).astype(int)
Etiquetas_penultimo_paso_sin_cod_uni.loc[Etiquetas_penultimo_paso_sin_cod_uni['Esta_En_histor_telec_TOTAL']==1, 'EstadoFinal'] = 'ENTRO CON OTRA FUERZA_T'

#VUELVO A UNIR MI LIQUIDACION
Etiquetas_penultimo_paso=[]
Etiquetas_penultimo_paso = pd.concat([Etiquetas_penultimo_Con_cod_uni, Etiquetas_penultimo_paso_sin_cod_uni], ignore_index=True)
#--------------------------------------------
#--------------------------------------------

In [88]:
# CREO UNA CONSULTA CON SOLO DONDE NUEVO = NUEVO Y CLASE ACTUAL= NUEVO
Consulta_recibos_per_Actual_solo_nuevos=df_recibos_per_Actual[(df_recibos_per_Actual['NUEVO']=='NUEVO')&(df_recibos_per_Actual['CLASE_ACTUAL']=='Nuevo')]
Consulta_recibos_per_Actual_solo_nuevos.shape

#AHORA SIMPLEMENTE COMPARO SI MIS LIQUIDACION ACTUAL EN SU RESPECTIVO PERIODO SI ES NUEVO
Etiquetas_penultimo_paso['Esta_en_consulta_Nuevos'] =Etiquetas_penultimo_paso['Llave_per'].isin(Consulta_recibos_per_Actual_solo_nuevos['llave_orden_sql']).astype(int)
Etiquetas_penultimo_paso['Esta_en_consulta_per_Actual'] =Etiquetas_penultimo_paso['Llave_per'].isin(df_recibos_per_Actual['llave_orden_sql']).astype(int)

condicion_real_antiguo=((Etiquetas_penultimo_paso['Esta_en_consulta_Nuevos']==0) & (Etiquetas_penultimo_paso['Esta_en_consulta_per_Actual']==1))
Etiquetas_penultimo_paso.loc[condicion_real_antiguo, 'EstadoFinal'] = 'Estudiante ANTIGUO_PER_ACT'

In [89]:
# CREO UNA CONSULTA DONDE HAYAN FUERZAS DE VENTA VALIDAS Y LUEGO DONDE SOLO SEA EL CONTACT
Consulta_recibos_per_Actual_varias_fuerzas=df_recibos_per_Actual.copy()
Consulta_recibos_per_Actual_varias_fuerzas=Consulta_recibos_per_Actual_varias_fuerzas[Consulta_recibos_per_Actual_varias_fuerzas['FuerzaComercialFinal']!= 'N/A']
Consulta_recibos_per_Actual_varias_fuerzas=Consulta_recibos_per_Actual_varias_fuerzas[Consulta_recibos_per_Actual_varias_fuerzas['FuerzaComercialFinal']!= 'NA']
Consulta_recibos_per_Actual_varias_fuerzas=Consulta_recibos_per_Actual_varias_fuerzas[Consulta_recibos_per_Actual_varias_fuerzas['FuerzaComercialFinal']!= 'PENDIENTE']
Consulta_recibos_per_Actual_varias_fuerzas=Consulta_recibos_per_Actual_varias_fuerzas[Consulta_recibos_per_Actual_varias_fuerzas['FuerzaComercialFinal'].notna()]

#Creo df solo con CONTACT
Consulta_recibos_per_Actual_solo_CONTACT=Consulta_recibos_per_Actual_varias_fuerzas[(Consulta_recibos_per_Actual_varias_fuerzas['FuerzaComercialFinal']=='CONTACT')|(Consulta_recibos_per_Actual_varias_fuerzas['FuerzaComercialFinal']=='CONTACT ')]

#AHORA SIMPLEMENTE COMPARO SI MIS LIQUIDACION ACTUAL EN SU RESPECTIVO PERIODO SI ES NUEVO
Etiquetas_penultimo_paso['Esta_en_consulta_CONTACT'] =Etiquetas_penultimo_paso['Llave_per'].isin(Consulta_recibos_per_Actual_solo_CONTACT['llave_orden_sql']).astype(int)
Etiquetas_penultimo_paso['Esta_en_consulta_per_Actual_total_fuerzas'] =Etiquetas_penultimo_paso['Llave_per'].isin(Consulta_recibos_per_Actual_varias_fuerzas['llave_orden_sql']).astype(int)

condicion_OTRA_fuerza=((Etiquetas_penultimo_paso['Esta_en_consulta_CONTACT']==0) & (Etiquetas_penultimo_paso['Esta_en_consulta_per_Actual_total_fuerzas']==1))
Etiquetas_penultimo_paso.loc[condicion_OTRA_fuerza, 'EstadoFinal'] = 'OTRA FUERZA DE VENTA_PER_ACT'

In [90]:
"Nuevamente Divido mi liquidacion entre los que si obtuvieron su cod_uni y los que no"
Etiquetas_penultimo_paso_sin_cod_uni=[]
Etiquetas_penultimo_Con_cod_uni=[]
Etiquetas_penultimo_paso_sin_cod_uni=Etiquetas_penultimo_paso[Etiquetas_penultimo_paso['Llave_liq_cod_uni'].isna()]
Etiquetas_penultimo_Con_cod_uni=Etiquetas_penultimo_paso[Etiquetas_penultimo_paso['Llave_liq_cod_uni'].notna()]

In [91]:
# AQUI SOLO ARMO UNA CONSULTA MAS GLOBAL no por periodo SINO con los CODIGOS DE unidad (contemplando todos los periodos) con los programas
# de la liquidacion
Consulta_total_cod_uni=df_ordenes[df_ordenes['llave_total_cod_uni'].isin(Etiquetas_penultimo_Con_cod_uni['Llave_liq_cod_uni'])]

# Dejo en la consultA
Consulta_total_cod_uni = Consulta_total_cod_uni.sort_values(by=['llave_total_cod_uni', 'PERIODO'], ascending=[True, True])
#------------------EXPORTANDO EVIDENCIA-------------------------------------
from datetime import datetime
fecha_hoy_rc = datetime.now().strftime('%Y-%m-%d')
Consulta_total_cod_uni.to_excel(f'Evidencia_Recibos_cont_CODUNI{fecha_hoy_rc}_Reliq.xlsx')
#-------------------------------------------------
Consulta_total_cod_uni = Consulta_total_cod_uni.drop_duplicates(subset='llave_total_cod_uni', keep='first')
# Justo en este momento tengo la consulta lista con los primeros periodos
Consulta_solo_para_validar_nuevo_por_uni=Consulta_total_cod_uni.copy()
#-------------------------------------------------
Consulta_total_cod_uni=Consulta_total_cod_uni[Consulta_total_cod_uni['FuerzaComercialFinal'] != 'N/A']
Consulta_total_cod_uni=Consulta_total_cod_uni[Consulta_total_cod_uni['FuerzaComercialFinal'] != 'NA']
Consulta_total_cod_uni=Consulta_total_cod_uni[Consulta_total_cod_uni['FuerzaComercialFinal'] != 'PENDIENTE']
Consulta_total_cod_uni=Consulta_total_cod_uni[Consulta_total_cod_uni['FuerzaComercialFinal'].notna()]

#creo una consulta por codigo de unidad donde de verdad estan solo CONTACT
Consulta_recibos_per_COD_UNI_solo_CONTACT=Consulta_total_cod_uni[(Consulta_total_cod_uni['FuerzaComercialFinal'] == 'CONTACT')|(Consulta_total_cod_uni['FuerzaComercialFinal'] == 'CONTACT ')]

#AHORA SIMPLEMENTE COMPARO SI MIS LIQUIDACION ACTUAL EN SU RESPECTIVO PERIODO SI ES NUEVO
Etiquetas_penultimo_Con_cod_uni['Esta_en_consulta_CONTACT_CODUNI'] =Etiquetas_penultimo_Con_cod_uni['Llave_liq_cod_uni'].isin(Consulta_recibos_per_COD_UNI_solo_CONTACT['llave_total_cod_uni']).astype(int)
Etiquetas_penultimo_Con_cod_uni['Esta_en_consulta_total_fuerzas_CODUNI'] =Etiquetas_penultimo_Con_cod_uni['Llave_liq_cod_uni'].isin(Consulta_total_cod_uni['llave_total_cod_uni']).astype(int)

condicion_OTRA_fuerza_CODUNI=((Etiquetas_penultimo_Con_cod_uni['Esta_en_consulta_CONTACT_CODUNI']==0) & (Etiquetas_penultimo_Con_cod_uni['Esta_en_consulta_total_fuerzas_CODUNI']==1) &(~Etiquetas_penultimo_Con_cod_uni['TIPO'].str.contains('continuada', case=False, na=False)))
Etiquetas_penultimo_Con_cod_uni.loc[condicion_OTRA_fuerza_CODUNI, 'EstadoFinal'] = 'OTRA FUERZA DE VENTA_COD_U'
#--------------------------------------------------

In [92]:
Consulta_solo_para_validar_nuevo_por_uni.rename(columns={'PERIODO': 'Primer_periodo_real'}, inplace=True)

In [93]:
#De la consulta sin duplicados por llave de codigo de unidad. Me traigo el primer periodo real a las etiquetas. (Aqui no se generan duplicados
# en las etiquetas)

Etiquetas_penultimo_Con_cod_uni = Etiquetas_penultimo_Con_cod_uni.merge(Consulta_solo_para_validar_nuevo_por_uni[['llave_total_cod_uni', 'Primer_periodo_real']], 
                                                left_on='Llave_liq_cod_uni', 
                                                right_on='llave_total_cod_uni',
                                                how='left')
Etiquetas_penultimo_Con_cod_uni.drop(columns=['llave_total_cod_uni'], inplace=True)
Etiquetas_penultimo_Con_cod_uni.shape
#----------------------------------------------
#----------------------------------------------

(54, 32)

In [94]:
#Si el periodo de la etiqueta no es igual al primer periodo real en el codigo de unidad =  ANTIGUO
condicion_antiguo_coduni=(
                (Etiquetas_penultimo_Con_cod_uni['Primer_periodo_real'].notna()) &
                (Etiquetas_penultimo_Con_cod_uni['PERIODO']!=Etiquetas_penultimo_Con_cod_uni['Primer_periodo_real']) &
                (~Etiquetas_penultimo_Con_cod_uni['TIPO'].str.contains('continuada', case=False, na=False)) 
)
Etiquetas_penultimo_Con_cod_uni.loc[condicion_antiguo_coduni, 'EstadoFinal'] = 'ANTIGUO_COD_U'

# Depuro columnas
Etiquetas_penultimo_Con_cod_uni.drop(columns=['Esta_en_consulta_Nuevos'], inplace=True)
Etiquetas_penultimo_Con_cod_uni.drop(columns=['Esta_en_consulta_per_Actual'], inplace=True)
Etiquetas_penultimo_Con_cod_uni.drop(columns=['Esta_en_consulta_CONTACT'], inplace=True)
Etiquetas_penultimo_Con_cod_uni.drop(columns=['Esta_en_consulta_per_Actual_total_fuerzas'], inplace=True)
Etiquetas_penultimo_Con_cod_uni.drop(columns=['Esta_en_consulta_CONTACT_CODUNI'], inplace=True)
Etiquetas_penultimo_Con_cod_uni.drop(columns=['Esta_en_consulta_total_fuerzas_CODUNI'], inplace=True)
Etiquetas_penultimo_Con_cod_uni.drop(columns=['Primer_periodo_real'], inplace=True)

Etiquetas_penultimo_Con_cod_uni.shape
#------------------------------------
#------------------------------------

(54, 25)

In [95]:
host2_3 = '172.16.1.33'
port2_3 = '1433'
database2_3 = 'CUN_REPOSITORIO'

try:
    url2_3 = 'mssql+pyodbc://@{host}:{port}/{db}?trusted_connection=yes&driver=SQL+Server'.format(host=host2_3, port=port2_3, db=database2_3)
    engine2_3 = create_engine(url2_3) 
    print("Conexion a la base de datos realizada")
    Lista_ref = pd.read_sql("select * from CUN_REPOSITORIO.Comisiones.HISTOR_REFERIDOS", engine2_3)
    Lista_ref2 = Lista_ref.copy()
        
except Exception as e:
    print("Error:", e)
finally:
    # Cerrar la conexión
    if 'engine2_3' in locals():
        engine2_3.dispose()
        print("Conexión cerrada")
Lista_ref

Conexion a la base de datos realizada
Conexión cerrada


,Listado_Referidos,Ejecucion
0,1000731646,_2024-10-23_
1,1047369554,_2024-10-23_
2,80016486,_2024-10-23_
3,1064193348,_2024-10-23_
4,1001115701,_2024-10-23_
...,...,...
2117,79895868,_2025_08_02_
2118,80257124,_2025_08_02_
2119,80750452,_2025_08_02_
2120,8128046,_2025_08_02_


In [96]:
#VUELVO A UNIR MI LIQUIDACION
Etiquetas_penultimo_paso=[]
Etiquetas_penultimo_paso = pd.concat([Etiquetas_penultimo_Con_cod_uni, Etiquetas_penultimo_paso_sin_cod_uni], ignore_index=True)
#--------------------------------
#--------------------------------

In [97]:
Lista_ref=Lista_ref.astype(str)
Etiquetas_penultimo_paso['Refer'] = Etiquetas_penultimo_paso['Doc_Estudiante'].isin(Lista_ref['Listado_Referidos']).astype(int)
Etiquetas_penultimo_paso.loc[Etiquetas_penultimo_paso['Refer'] == 1, 'EstadoFinal'] = 'OK'

In [98]:
host2_1 = '172.16.1.33'
port2_1 = '1433'
database2_1 = 'CUN_REPOSITORIO'

try:
    url2_1 = 'mssql+pyodbc://@{host}:{port}/{db}?trusted_connection=yes&driver=SQL+Server'.format(host=host2_1, port=port2_1, db=database2_1)
    engine2_1 = create_engine(url2_1) 
    print("Conexion a la base de datos realizada")
    cedulas_erroneas= pd.read_sql(f'''
select * from CUN_REPOSITORIO.JUAN_GUZMAN.CEDULAS_NO_LIQUIDAR
                            ''', engine2_1)
    cedulas_erroneas_copia =  cedulas_erroneas.copy()
    cedulas_erroneas3= pd.read_sql(f'''
    SELECT Documento_Estudiante_zoho, Periodo, FuerzaComercialFinal AS Fuerza_comercial
FROM CUN_REPOSITORIO.DBO.VENTAS_REINTEGROS
UNION ALL
SELECT Documento_Estudiante_zoho, Periodo, FuerzaComercialFinal
FROM CUN_REPOSITORIO.DBO.VENTAS_REINTEGROS_a
UNION ALL
SELECT Documento_Estudiante_zoho, Periodo, FuerzaComercialFinal
FROM CUN_REPOSITORIO.DBO.VENTAS_REINTEGROS_25V02
UNION ALL
SELECT Documento_Estudiante_zoho, Periodo, FuerzaComercialFinal
FROM CUN_REPOSITORIO.DBO.VENTAS_REINTEGROS_2025B
UNION ALL
SELECT Documento_Estudiante_zoho, Periodo, FuerzaComercialFinal
FROM CUN_REPOSITORIO.DBO.VENTAS_REINTEGROS_25V02_s
UNION ALL
SELECT Documento_Estudiante_zoho, Periodo, FuerzaComercialFinal
FROM CUN_REPOSITORIO.DBO.VENTAS_BASICA_MEDIA
UNION ALL
SELECT Documento_Estudiante_zoho, Periodo, FuerzaComercialFinal
FROM CUN_REPOSITORIO.DBO.VENTAS25V02_contact
UNION ALL
SELECT Documento_Estudiante_zoho, Periodo, FuerzaComercialFinal
FROM CUN_REPOSITORIO.DBO.VENTAS25V02_contact_2
UNION ALL
SELECT Documento_Estudiante_zoho, Periodo, FuerzaComercialFinal
FROM CUN_REPOSITORIO.DBO.VENTAS_REINTEGROS_25V02_2
UNION ALL
SELECT Documento_Estudiante_zoho, Periodo, FuerzaComercialFinal
FROM CUN_REPOSITORIO.DBO.VENTAS_REINTEGROS_2025B_2
UNION ALL
SELECT Documento_Estudiante_zoho, Periodo, FuerzaComercialFinal
FROM CUN_REPOSITORIO.DBO.VENTAS_REINTEGROS_25V02_3
UNION ALL
SELECT Documento_Estudiante_zoho, Periodo, FuerzaComercialFinal
FROM CUN_REPOSITORIO.DBO.VENTAS_REINTEGROS_25V02_4
UNION ALL
SELECT Documento_Estudiante_zoho, Periodo, FuerzaComercialFinal
FROM CUN_REPOSITORIO.DBO.VENTAS_REINTEGROS_2025B_3;
                            ''', engine2_1)
        
except Exception as e:
    print("Error:", e)
finally:
    # Cerrar la conexión
    if 'engine2_1' in locals():
        engine2_1.dispose()
        print("Conexión cerrada")
cedulas_erroneas['Cliente']=cedulas_erroneas['Documento_Estudiante_zoho'].astype(float).astype(int).astype(str)
'-----------------------------------------------------------------------------------------'
'-----------------------------------------------------------------------------------------'
cedulas_erroneas2 = pd.read_excel("EMORENO_Cedulas_reintegros_2025_ENERO.xlsx")
cedulas_erroneas2['Cliente']=cedulas_erroneas2['Cliente'].astype(float).astype(int).astype(str)
'-----------------------------------------------------------------------------------------'
'-----------------------------------------------------------------------------------------'
cedulas_erroneas = pd.concat([cedulas_erroneas, cedulas_erroneas2, cedulas_erroneas3], ignore_index=True)
cedulas_erroneas = cedulas_erroneas.drop_duplicates()
Etiquetas_penultimo_paso['Reint_Stv'] = Etiquetas_penultimo_paso['Doc_Estudiante'].isin(cedulas_erroneas['Cliente']).astype(int)
Etiquetas_penultimo_paso.loc[Etiquetas_penultimo_paso['Reint_Stv'] == 1, 'EstadoFinal'] = 'OTRA_FUERZA_stv'
#---------------------------
#---------------------------

Conexion a la base de datos realizada
Conexión cerrada


In [99]:
df_filtrado_1=Etiquetas_penultimo_paso.copy()

In [100]:
host2_2 = '172.16.1.33'
port2_2 = '1433'
database2_2= 'CUN_REPOSITORIO'

try:
    url2_2 = 'mssql+pyodbc://@{host}:{port}/{db}?trusted_connection=yes&driver=SQL+Server'.format(host=host2_2, port=port2_2, db=database2_2)
    engine2_2= create_engine(url2_2) 
    print("Conexion a la base de datos realizada")
    archivo_contact = pd.read_sql(f'''
select * from CUN_REPOSITORIO.comisiones_contact.etiquetas where EstadoFinal='OK'
                                  ''', engine2_2)
    archivo_contact2 = archivo_contact.copy()
        
except Exception as e:
    print("Error:", e)
finally:
    # Cerrar la conexión
    if 'engine2_2' in locals():
        engine2_2.dispose()
        print("Conexión cerrada")
archivo_contact.shape

Conexion a la base de datos realizada
Conexión cerrada


(34105, 20)

In [101]:
host2_4 = '172.16.1.33'
port2_4 = '1433'
database2_4= 'CUN_REPOSITORIO'

try:
    url2_4 = 'mssql+pyodbc://@{host}:{port}/{db}?trusted_connection=yes&driver=SQL+Server'.format(host=host2_4, port=port2_4, db=database2_4)
    engine2_4= create_engine(url2_4) 
    print("Conexion a la base de datos realizada")
    archivo_contact3 = pd.read_sql(f'''
select * from CUN_REPOSITORIO.comisiones_contact.EtiquetasReliquidacion where EstadoFinal='OK' and Periodo_Pago <>{MES_ACTUAL_EN_RELIQUIDACION}                                   
                                   ''', engine2_4)
    archivo_contact4 = archivo_contact3.copy()
        
except Exception as e:
    print("Error:", e)
finally:
    # Cerrar la conexión
    if 'engine2_4' in locals():
        engine2_4.dispose()
        print("Conexión cerrada")

Conexion a la base de datos realizada
Conexión cerrada


In [102]:
host2_4 = '172.16.1.33'
port2_4 = '1433'
database2_4= 'CUN_REPOSITORIO'

try:
    url2_4 = 'mssql+pyodbc://@{host}:{port}/{db}?trusted_connection=yes&driver=SQL+Server'.format(host=host2_4, port=port2_4, db=database2_4)
    engine2_4= create_engine(url2_4) 
    print("Conexion a la base de datos realizada")
    CONTACT_UNICD = pd.read_sql('''
select * from CUN_REPOSITORIO.comisiones_CONTACT.CONTACT_UNICD
WHERE COD_UNI IS NOT NULL
                                ''', engine2_4)
        
except Exception as e:
    print("Error:", e)
finally:
    # Cerrar la conexión
    if 'engine2_4' in locals():
        engine2_4.dispose()
        print("Conexión cerrada")

Conexion a la base de datos realizada
Conexión cerrada


In [103]:
archivo_contact2 = pd.concat([archivo_contact2, archivo_contact4], ignore_index=True)
archivo_contact2['Doc_Estudiante']=archivo_contact2['Doc_Estudiante'].astype(str)

In [104]:
archivo_contact2=archivo_contact2[archivo_contact2['Doc_Estudiante'].isin(df_filtrado_1['Doc_Estudiante'])]
archivo_contact2['Llave_etiq_cont']=archivo_contact2['Doc_Estudiante']+archivo_contact2['PERIODO']
# Quito duplicados de CONTACT_UNICD para añadir el cod_uni a las etiquetas
CONTACT_UNICD = CONTACT_UNICD.drop_duplicates(subset='llave_per', keep='first')

archivo_contact2 = archivo_contact2.merge(CONTACT_UNICD[['llave_per', 'COD_UNI']], 
                                     left_on='Llave_etiq_cont', 
                                     right_on='llave_per', 
                                     how='left')
archivo_contact2.rename(columns={'COD_UNI': 'COD_UNI_CONTACT'}, inplace=True)
archivo_contact2.drop(columns=['llave_per'], inplace=True)
#-----------------------------------------------
#-----------------------------------------------

In [105]:
df_filtrado_1['Esta_en_hist_Cont_x_per']=df_filtrado_1['Llave_per'].isin(archivo_contact2['Llave_etiq_cont']).astype(int)
df_filtrado_1.loc[df_filtrado_1['Esta_en_hist_Cont_x_per']==1, 'EstadoFinal'] = 'YA SE PAGO'
#----------------------------------

In [106]:
"Nuevamente Divido mi liquidacion entre los que si obtuvieron su cod_uni y los que no"

df_final_sin_uni=df_filtrado_1[df_filtrado_1['Llave_liq_cod_uni'].isna()]
df_filtrado_1=df_filtrado_1[df_filtrado_1['Llave_liq_cod_uni'].notna()]

In [107]:
def comparar_y_agregar_pert_contact(df_final, archivo_contact2):
    # Iterar sobre cada fila de df_final
    for idx, row in df_final.iterrows():
        # Obtener el valor de 'DOC_ALUM', 'COD_UNI' y 'EstadoFinal' de la fila actual de df_final
        doc_alum = row['Doc_Estudiante']
        cod_uni = row['COD_UNI']
        estado_fini = row['EstadoFinal']
        
        # Buscar en archivo_contact2 si existe alguna fila que tenga 'Doc_Estudiante' igual a 'DOC_ALUM'
        match = archivo_contact2[archivo_contact2['Doc_Estudiante'] == doc_alum]
        
        # Si no se encuentra coincidencia en 'Doc_Estudiante'
        if match.empty:
            df_final.at[idx, 'EstadoFinal'] = estado_fini
        else:
            # Si se encuentra coincidencia, comprobar la columna 'COD_UNI_CONTACT'
            cod_uni_contact = match['COD_UNI_CONTACT'].values[0]
            
            if pd.isna(cod_uni_contact):  # Si 'COD_UNI_CONTACT' es NaN
                df_final.at[idx, 'EstadoFinal'] = 'YA SE PAGO'
            elif cod_uni == cod_uni_contact:  # Si 'COD_UNI' coincide con 'COD_UNI_CONTACT'
                df_final.at[idx, 'EstadoFinal'] = 'YA SE PAGO'
            else:  # Si no coincide
                df_final.at[idx, 'EstadoFinal'] = estado_fini
    
    return df_final
#--------------------------------------------------------

In [108]:
df_filtrado_1 = comparar_y_agregar_pert_contact(df_filtrado_1, archivo_contact2)
#------------------------------------

In [109]:
df_final_sin_uni['Esta_en_hist_cont'] = df_final_sin_uni['Doc_Estudiante'].isin(archivo_contact2['Doc_Estudiante']).astype(int)
df_final_sin_uni.loc[df_final_sin_uni['Esta_en_hist_cont'] == 1, 'EstadoFinal'] = 'YA SE PAGO'

In [110]:
df_filtrado_1 = pd.concat([df_filtrado_1, df_final_sin_uni], ignore_index=True)

In [111]:
df_filtrado_1.rename(columns={'Llave_per': 'llave_per'}, inplace=True)
CONTAC_UNI_TEMP=df_filtrado_1.copy()
CONTAC_UNI_TEMP['Fecha_de_pago_Real']="-"
CONTAC_UNI_TEMP['Fecha_de_pago']="-"
CONTAC_UNI_TEMP=CONTAC_UNI_TEMP[['Doc_Estudiante',	'PERIODO',	'Documento_del_Asesor',	'Fecha_de_pago_Real',	'Propietario_del_registro',	
                             'Programa_vendido',	'Fecha_de_pago',	'Correo_Asesor',	'Periodo_Pago',	'EST_MATRICULADO',	
                             'NCR_ANULACION',	'TIPO',	'ORDEN',	'VAL_LIQUIDADO',	'NUEVO',	'ESTADO_PAGO',	'Documento_Estudiante_zoho',
                             'Motivo',	'Propietario_de_Fase_Name',	'EstadoFinal',	'llave_per',	'COD_UNI']]
#-----------------------

In [112]:
host_2_12 = '172.16.1.33'
port_2_12 = '1433'
database_2_12 = 'CUN_REPOSITORIO'
url_2_12 = 'mssql+pyodbc://@{host}:{port}/{db}?driver=ODBC Driver 17 for SQL Server'.format(host=host_2_12, port=port_2_12, db=database_2_12)
engine_2_12 = create_engine(url_2_12, fast_executemany=True)

table_name_2_12 = 'CONTACT_UNICD'
schema_name_2_12 = 'COMISIONES_CONTACT'
#Send the data to sql server
CONTAC_UNI_TEMP.to_sql(
     name=table_name_2_12,
     schema=schema_name_2_12,
     con=engine_2_12,
     if_exists='append',
     index=False,
     chunksize = 10000
 )

# # End connection
engine_2_12.dispose()

In [113]:
df_filtrado_1=df_filtrado_1[['Doc_Estudiante',	'PERIODO',	'Documento_del_Asesor',	'Propietario_del_registro',	'Programa_vendido',	
                             'Correo_Asesor',	'Periodo_Pago',	'EST_MATRICULADO',	'NCR_ANULACION',	'TIPO',	'ORDEN',	'VAL_LIQUIDADO',	
                             'NUEVO',	'ESTADO_PAGO',	'Documento_Estudiante_zoho',	'Motivo',	'Propietario_de_Fase_Name',	'EstadoFinal'
                            ]]
#--------------------
#--------------------

In [114]:
host_2_12 = '172.16.1.33'
port_2_12 = '1433'
database_2_12 = 'CUN_REPOSITORIO'
url_2_12 = 'mssql+pyodbc://@{host}:{port}/{db}?driver=ODBC Driver 17 for SQL Server'.format(host=host_2_12, port=port_2_12, db=database_2_12)
engine_2_12 = create_engine(url_2_12, fast_executemany=True)

table_name_2_12 = 'EtiquetasReliquidacion'
schema_name_2_12 = 'COMISIONES_CONTACT'
# # Send the data to sql server
df_filtrado_1.to_sql(
     name=table_name_2_12,
     schema=schema_name_2_12,
     con=engine_2_12,
     if_exists='append',
     index=False,
     chunksize = 10000
 )

# # End connection
engine_2_12.dispose()

In [115]:
#CONSULTA PARA CREAR LA TABLA TEMPORAL EN SQL 
# Busca en tabla Base pago mes anterior, y cruza contra las reliquidadas para sacar la diferencia 
with engine.connect() as connection:
    try:

        sql_query = text(f'''
        IF object_id('CUN_REPOSITORIO.COMISIONES_CONTACT.TEMP_BaseReliquidacionCal') IS NOT NULL
                DROP TABLE CUN_REPOSITORIO.COMISIONES_CONTACT.TEMP_BaseReliquidacionCal

            Select

                distinct t1.*,
                        isnull(t2.Pregrados_Reliquidacion,0)Pregrados_Reliquidacion,
                        isnull(t2.Postgrado_Reliquidacion,0)Postgrado_Reliquidacion,
                        isnull(t2.Educacion_continuada_Reliquidacion,0)Educacion_continuada_Reliquidacion
            INTO CUN_REPOSITORIO.COMISIONES_CONTACT.TEMP_BaseReliquidacionCal
                from Comisiones_Contact.BasePago t1
                left join (select distinct Documento_del_Asesor,Periodo_Pago,
                sum(case when TIPO = 'Pregrado' then 1 else 0 end) As Pregrados_Reliquidacion,
                sum(case when TIPO = 'Postgrado' then 1 else 0 end) As Postgrado_Reliquidacion,
                sum(case when TIPO = 'Educacion continuada' then 1 else 0 end) As Educacion_continuada_Reliquidacion
                from Comisiones_Contact.EtiquetasReliquidacion where EstadoFinal = 'OK'
                                                                and Periodo_Pago = {MES_ACTUAL_EN_RELIQUIDACION}
                group by Documento_del_Asesor,Periodo_Pago) t2
                                                        on t1.Documento_del_Asesor = t2.Documento_del_Asesor
                where t1.Periodo_Pago = {MES_ACTUAL_EN_RELIQUIDACION}

        ''')
        connection.execute(sql_query)
        connection.commit()  # Asegurar que los cambios se guarden en la base de datos
        print("Consulta realizada con exito")
    except Exception as e:
        print(f"Error al eliminar registros: {e}")
    finally:
        connection.close()

Consulta realizada con exito


In [116]:
##Realiza los calculos de la liquidacion del mes pasado contra reliquidacion, carga en tabla BasePagoreliquidaciones 
with engine.connect() as connection:
    try:

        sql_query = text(f'''
        Delete from Comisiones_Contact.BasePagoReliquidaciones where Periodo_Pago = {MES_ACTUAL_EN_RELIQUIDACION}
        delete from Comisiones_Contact.RELIQ_PREGR_APROB_CONTAC_MOD2 where Periodo_Pago_1 = {MES_ACTUAL_EN_RELIQUIDACION}
        delete from Comisiones_Contact.RELIQ_SOL_PREG_X where Periodo_Pago_1 = {MES_ACTUAL_EN_RELIQUIDACION}
        delete from Comisiones_Contact.RELIQ_DESGLO_BASE where Per_desglo = {MES_ACTUAL_EN_RELIQUIDACION}
          
        ''')
        connection.execute(sql_query)
        connection.commit()  # Asegurar que los cambios se guarden en la base de datos
        print("Consulta realizada con exito")
    except Exception as e:
        print(f"Error al eliminar registros: {e}")
    finally:
        connection.close()

Consulta realizada con exito
